# M10: CI/CD (Continuous Integration / Continuous Deployment) for Data Engineering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/M10_CICD_for_Data_Engineering.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View_on_GitHub-blue?logo=github)](https://github.com/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/M10_CICD_for_Data_Engineering.ipynb)

**Program:** Quintrix Jr. Data Engineer Training
**Author:** Sunil Mogadati
**Prerequisites:** M08 (Cloud Data Pipeline — Build), M09 (Cloud Data Pipeline — Orchestrate)
**Estimated Time:** 6–8 hours across 2 sessions

---

### What You'll Build

By the end of this module you will have:

- Read and understood Terraform HCL (HashiCorp Configuration Language) configs you'll encounter at every company
- Written Python (CDKTF (CDK for Terraform)) to provision your entire GCP pipeline infrastructure
- Written Python (AWS CDK) to provision the same pipeline on AWS
- Built a GitHub Actions CI workflow that lints, tests, and validates your pipeline before merge
- Automated DAG (Directed Acyclic Graph) deployment so merging a PR pushes your Airflow DAG to Cloud Composer / MWAA
- Added a data quality gate that blocks bad transforms from reaching production

---

### Quick Reference: What You Already Have

| Asset | Location | Built In |
|---|---|---|
| `bronze_ingest.py` | GCS (Google Cloud Storage) / S3 (Simple Storage Service) | M08 |
| `silver_transform.py` | GCS / S3 | M08 |
| `va_analytics_pipeline` DAG | Cloud Composer / MWAA | M09 |
| GCS bucket, Dataproc cluster, BigQuery datasets | GCP Console | M08 |
| S3 bucket, EMR (Elastic MapReduce) cluster, Redshift cluster | AWS Console | M08 |

Today you will **codify all of that infrastructure** so it can be re-created from a single command.

---
## Table of Contents

| # | Section | Format | Time |
|---|---------|--------|------|
| 1 | [Why CI/CD for Data Pipelines](#1-why-cicd) | Lecture | 20 min |
| 2 | [Infrastructure as Code — The Landscape](#2-iac-landscape) | Lecture | 25 min |
| 3 | [Terraform HCL — Read and Understand](#3-terraform-hcl) | Lecture + Demo | 30 min |
| 4 | [CDKTF — Hands-On Deep Dive (Python)](#4-cdktf) | Lab | 60 min |
| 5 | [AWS CDK — Same Pipeline on AWS](#5-aws-cdk) | Lab | 40 min |
| 6 | [GitHub Actions — CI Pipeline](#6-github-actions) | Lab | 50 min |
| 7 | [Environment Promotion: dev → staging → prod](#7-environments) | Lecture + Demo | 30 min |
| 8 | [Data Quality Gates in CI](#8-quality-gates) | Lab | 30 min |
| 9 | [DAG Deployment — GitOps for Airflow](#9-dag-deployment) | Lab | 30 min |
| — | [Key Takeaways + Homework](#takeaways) | — | 10 min |

---
## Before We Start: Why This Module Matters to the Business

In M08 you built a pipeline. In M09 you automated it with Airflow. It runs every morning without anyone touching it.

**But here is what can still go wrong:**

| Day | What Happens |
|-----|-------------|
| Monday | Developer changes a column name in `silver_transform.py` |
| Monday | Developer pushes directly to Dataproc production |
| Monday night | Gold mart `fact_campaign_performance` breaks silently |
| Tuesday morning | VP opens dashboard — revenue numbers are wrong |
| Tuesday | Analyst sends report to client with wrong numbers |
| Wednesday | Data engineer gets paged — spends 4 hours diagnosing |
| Wednesday afternoon | Root cause: one column rename. Fix takes 3 minutes. |

**The cost of that one unreviewed push:** 4 engineering hours + wrong client report + trust damage.

CI/CD exists to make this scenario impossible. Every change goes through inspection before reaching production. Automated. Every time. No exceptions.

---
## 1. Why CI/CD for Data Pipelines <a id="1-why-cicd"></a>

> **In plain English:** CI/CD is like a car factory's quality control line. Every car (code change) goes through inspection (CI) before reaching the showroom floor (CD). A mechanic can make a mistake — that's normal. What the factory prevents is a defective car reaching the customer. Without CI/CD, the developer's mistake goes straight to production. The VP sees the wrong revenue. The mechanic never knows until someone calls.

---

### What CI/CD Stands For

| Term | Full Name | What It Means for Data |
|---|---|---|
| **CI** | Continuous Integration | Every code change is automatically tested before merge |
| **CD** | Continuous Delivery / Deployment | Tested code is automatically deployed to the target environment |

---

### "But I'm a Data Engineer, Not a Software Developer"

This is the most common misconception in the field. Look at everything you wrote in M08 and M09:

| What You Wrote | It Is Code |
|---|---|
| `bronze_ingest.py` | Python application code |
| `silver_transform.py` | PySpark application code |
| `va_analytics_pipeline.py` (DAG) | Python workflow code |
| BigQuery SQL transforms | SQL code |
| GCS bucket config | Infrastructure config |
| Dataproc cluster settings | Infrastructure config |

**All of it needs version control, review, and automated testing.** A broken SQL transform causes the same damage as a broken API endpoint. The VP doesn't care which kind of code broke the dashboard.

---

### What Goes Wrong Without CI/CD

```
Developer changes silver_transform.py locally
  ↓
Tests it on 3 rows — looks fine
  ↓
Copies the file directly to Dataproc via gcloud scp
  ↓
Airflow runs at 6 AM — transform fails on full dataset
  ↓
Gold marts show NULL revenue for all campaigns
  ↓
Morning dashboard shows $0 in Q1 revenue
  ↓
VP sends emergency email at 7:30 AM
  ↓
On-call engineer spends 4 hours finding the 1-line bug
```

**With CI/CD:**

```
Developer opens Pull Request
  ↓
GitHub Actions runs automatically:
  - Lint check (catches syntax errors)
  - Unit tests (catches logic errors)
  - Schema validation (catches column renames)
  - DAG import check (catches import errors)
  ↓
All checks pass → Reviewer approves → Merge
  ↓
CD automatically deploys to staging → then prod
  ↓
Nobody is paged at 7:30 AM
```

In [ ]:
# ============================================================
# SECTION 1 -- Simulating the "no CI/CD" failure scenario
# CI = Continuous Integration (automated checks that run on every code push)
# CD = Continuous Deployment (automated pipeline that ships code to production)
# ============================================================
# WHY this cell exists: before learning CI/CD tools, you need to feel
# the pain of NOT having them. This simulation shows how a single
# column rename silently wipes out conversion metrics in the Gold mart.
# Everything runs in-memory -- no cloud access needed.

from datetime import datetime, date
import random

# WHY seed the random number generator?
# Setting a fixed seed makes the demo reproducible -- every run produces
# the same 20 rows, so your output always matches the expected numbers.
random.seed(99)

# ─────────────────────────────────────────────────────────────────────────────
# HELLO-WORLD RAMP: the raw Bronze data we are working with
# A call record has: ID, date, duration in seconds, outcome, and campaign key.
# In production this comes from the LT (ListenTrust) call center platform.
# ─────────────────────────────────────────────────────────────────────────────

# BEGINNER NOTE: list comprehension with f-strings -- this is equivalent to a
# for-loop that appends to a list. f"C{i:04d}" zero-pads i to 4 digits
# so C1 becomes C0001, keeping sort order consistent.
CALLS_SAMPLE = [
    {"call_id": f"C{i:04d}", "call_date": "2026-03-20",
     "duration_sec": random.randint(30, 600),
     "outcome": random.choice(["converted", "not_converted", "callback"]),
     "campaign_key": random.randint(1, 5)}
    for i in range(1, 21)
]

# You should see: 20 rows in memory -- no database needed for this demo.
print(f"Sample size: {len(CALLS_SAMPLE)} call records")

# ─────────────────────────────────────────────────────────────────────────────
# Silver transform -- version 1 (correct)
# Silver = middle layer of the medallion architecture (Bronze -> Silver -> Gold).
# Here we clean the raw seconds value and encode the outcome as a 0/1 flag.
# ─────────────────────────────────────────────────────────────────────────────
def silver_transform_v1(calls):
    """Original Silver transform -- uses outcome column."""
    return [
        {
            "call_id": r["call_id"],
            "call_date": r["call_date"],
            # WHY divide by 60? Analysts prefer minutes over seconds for
            # readability in dashboards and SQL GROUP BY queries.
            "duration_min": round(r["duration_sec"] / 60, 2),
            # WHY encode as 0/1? Numeric flags let SQL SUM() count conversions
            # without a WHERE clause -- simpler Gold aggregations.
            "converted": 1 if r["outcome"] == "converted" else 0,
            "campaign_key": r["campaign_key"],
        }
        for r in calls
    ]

# ─────────────────────────────────────────────────────────────────────────────
# Silver transform -- version 2 (broken: developer renamed outcome column)
# WHY is this realistic? Developers often rename columns during refactoring
# without realising downstream jobs depend on the old name. Without CI/CD,
# this ships to production and silently breaks the Gold mart.
# ─────────────────────────────────────────────────────────────────────────────
def silver_transform_v2_broken(calls):
    """Developer renamed outcome to call_result -- forgot to update Gold."""
    return [
        {
            "call_id": r["call_id"],
            "call_date": r["call_date"],
            "duration_min": round(r["duration_sec"] / 60, 2),
            # BREAKING CHANGE: call_result replaces converted.
            # The Gold mart still looks for converted -- silently gets 0.
            "call_result": r["outcome"],   # <-- renamed column!
            "campaign_key": r["campaign_key"],
        }
        for r in calls
    ]

# ─────────────────────────────────────────────────────────────────────────────
# Gold mart aggregation -- depends on the Silver schema being stable
# Gold = the analyst-facing layer; aggregates Silver into per-campaign KPIs.
# ─────────────────────────────────────────────────────────────────────────────
def build_gold_mart(silver_rows):
    """Gold mart aggregation -- expects the 'converted' column."""
    mart = {}
    for row in silver_rows:
        # Group by (campaign_key, call_date) -- the grain of the Gold mart
        k = (row["campaign_key"], row["call_date"])
        if k not in mart:
            mart[k] = {"calls": 0, "conversions": 0}
        mart[k]["calls"] += 1
        # WHY .get("converted", 0)?
        # dict.get() returns the default (0) when the key is missing instead of
        # raising a KeyError. This is the silent-failure bug: when v2_broken
        # ships, the missing converted key produces 0 with no exception.
        mart[k]["conversions"] += row.get("converted", 0)
    return mart

# ─────────────────────────────────────────────────────────────────────────────
# Scenario A: Before the bad push -- correct data flows end-to-end
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("  SCENARIO: No CI/CD -- developer pushes directly to prod")
print("=" * 60)

silver_v1 = silver_transform_v1(CALLS_SAMPLE)
gold_v1 = build_gold_mart(silver_v1)
total_conversions_v1 = sum(v["conversions"] for v in gold_v1.values())

# You should see: non-zero conversion count and 5 Gold campaigns (keys 1-5)
print(f"\nBefore the bad push:")
print(f"  Silver rows: {len(silver_v1)}")
print(f"  Gold campaigns: {len(gold_v1)}")
print(f"  Total conversions in Gold mart: {total_conversions_v1}")

# ─────────────────────────────────────────────────────────────────────────────
# Scenario B: After the bad push -- Gold mart silently reports 0 conversions
# ─────────────────────────────────────────────────────────────────────────────
silver_v2 = silver_transform_v2_broken(CALLS_SAMPLE)
gold_v2 = build_gold_mart(silver_v2)
total_conversions_v2 = sum(v["conversions"] for v in gold_v2.values())

# You should see: 0 conversions -- the VP dashboard is now wrong.
print(f"\nAfter the bad push (column renamed, Gold mart unchanged):")
print(f"  Silver rows: {len(silver_v2)}")
print(f"  Gold campaigns: {len(gold_v2)}")
print(f"  Total conversions in Gold mart: {total_conversions_v2}  <- WRONG")

print(f"\nDamage: VP sees {total_conversions_v2} conversions instead of {total_conversions_v1}")
print("  That's a {:.0f}% data loss -- silently.".format(
    (1 - total_conversions_v2 / max(total_conversions_v1, 1)) * 100))

# ─────────────────────────────────────────────────────────────────────────────
# What CI would have caught -- schema diff between v1 and v2
# A CI (Continuous Integration) step can compare the actual output columns
# against the expected schema and FAIL the PR before the code ships.
# PR = Pull Request (a code-review step where a developer proposes a change)
# ─────────────────────────────────────────────────────────────────────────────
print()
print("What a schema validation CI step would have caught:")
v1_cols = set(silver_v1[0].keys())
v2_cols = set(silver_v2[0].keys())
added   = v2_cols - v1_cols    # new columns that appeared
removed = v1_cols - v2_cols    # columns that disappeared (dangerous!)
print(f"  Columns added:   {added}")
print(f"  Columns removed: {removed}")
print("  CI would FAIL here -> PR blocked -> bug never reaches production")

---
## 2. Infrastructure as Code — The Landscape <a id="2-iac-landscape"></a>

> **In plain English:** IaC (Infrastructure as Code) is the blueprint for a building. You don't tell the construction crew "build something like last time — you know, the one with the big lobby." You hand them a printed blueprint. If you need another identical building in another city, you hand that crew the same blueprint. The building comes out identical. Infrastructure as Code is the same idea: your GCS bucket, Dataproc cluster, and BigQuery datasets are described in a file. Anyone can run that file and get the exact same infrastructure.

---

### Console vs CLI (Command Line Interface) vs IaC

| Approach | How You Work | Problem |
|---|---|---|
| **Console (point-and-click)** | Log into GCP, click buttons, fill forms | No record of what you did. Can't reproduce. Can't review. |
| **CLI** | `gcloud dataproc clusters create ...` | Reproducible, but a long shell script with no state management |
| **IaC** | Describe desired state in a file. Tool figures out how to get there. | Small learning curve — but solves every problem above |

**IaC tools manage state.** They know what exists, what needs to change, and what to destroy. CLI scripts don't.

---

### The Four IaC Tools You Need to Know

| Tool | Language | Cloud | Output | Best For |
|---|---|---|---|---|
| **Terraform HCL** | HCL (its own language) | Multi-cloud | Direct API calls | Industry standard. You will read this everywhere. |
| **CDKTF** | Python / TypeScript | Multi-cloud (via Terraform providers) | Terraform JSON → API calls | Python-first teams who want Terraform's ecosystem |
| **AWS CDK** | Python / TypeScript | AWS only | CloudFormation YAML (YAML Ain't Markup Language) → API calls | AWS shops using Python |
| **Pulumi** | Python / TypeScript | Multi-cloud | Direct API calls | Python-first, no Terraform dependency |

---

### Decision Framework: Which Tool to Use?

```
Is the team AWS-only?
  ├─ YES → AWS CDK (native AWS, Python, great DX)
  └─ NO (multi-cloud or GCP-primary)
       ├─ Does the team already use Terraform?
       │    ├─ YES → CDKTF (reuse Terraform providers, write Python)
       │    └─ NO → Pulumi (pure Python, no Terraform layer) OR CDKTF
       └─ Need to READ existing configs?
            → Learn Terraform HCL (it's everywhere)
```

**For this cohort:**
- GCP primary → **CDKTF** for hands-on lab (Section 4)
- AWS secondary → **AWS CDK** for comparison lab (Section 5)
- Reading configs at any company → **Terraform HCL** (Section 3)

---
## 3. Terraform HCL — Read and Understand <a id="3-terraform-hcl"></a>

> **In plain English:** HCL (HashiCorp Configuration Language) is Terraform's native language. It looks like a simplified version of JSON with extra readability. You will see Terraform configs at **every data engineering company** — in runbooks, in pull requests, in job interviews. You don't need to master HCL. You need to read it confidently.

---

### Key Terraform Concepts

| Concept | What It Is | Analogy |
|---|---|---|
| **Provider** | Plugin that talks to GCP / AWS / Azure API | The translator between Terraform and the cloud |
| **Resource** | A cloud thing you want to create | A building on the blueprint |
| **Variable** | Input parameter | A blank you fill in on the form |
| **Output** | Value Terraform prints after apply | The receipt after checkout |
| **State** | Terraform's memory of what it created | The as-built drawing after construction |
| **Plan** | Preview of what Terraform will do | The contractor's quote before work starts |
| **Apply** | Execute the plan | The contractor does the work |

---

### The Terraform Workflow

```
terraform init     # Download providers (like pip install)
terraform plan     # Preview changes (dry run)
terraform apply    # Create/update/delete resources
terraform destroy  # Tear everything down
```

---

### Our Pipeline as Terraform HCL — GCP

```hcl
# provider.tf
terraform {
  required_providers {
    google = {
      source  = "hashicorp/google"
      version = "~> 5.0"
    }
  }
}

provider "google" {
  project = var.project_id
  region  = var.region
}

# variables.tf
variable "project_id" {
  description = "GCP project ID"
  type        = string
}

variable "region" {
  description = "GCP region"
  type        = string
  default     = "us-central1"
}

variable "env" {
  description = "Environment: dev / staging / prod"
  type        = string
  default     = "dev"
}

# main.tf — GCS Buckets (Bronze / Silver / Gold)
resource "google_storage_bucket" "bronze" {
  name          = "${var.project_id}-bronze-${var.env}"
  location      = var.region
  force_destroy = var.env != "prod"

  lifecycle_rule {
    condition { age = 90 }
    action    { type = "Delete" }
  }
}

resource "google_storage_bucket" "silver" {
  name     = "${var.project_id}-silver-${var.env}"
  location = var.region
}

resource "google_storage_bucket" "gold" {
  name     = "${var.project_id}-gold-${var.env}"
  location = var.region
}

# Dataproc cluster
resource "google_dataproc_cluster" "pipeline" {
  name   = "va-pipeline-${var.env}"
  region = var.region

  cluster_config {
    master_config {
      num_instances = 1
      machine_type  = "n1-standard-4"
    }
    worker_config {
      num_instances = 2
      machine_type  = "n1-standard-4"
    }
    preemptible_worker_config {
      num_instances = var.env == "prod" ? 4 : 0
    }
  }
}

# BigQuery datasets
resource "google_bigquery_dataset" "bronze" {
  dataset_id = "bronze_${var.env}"
  location   = "US"
}

resource "google_bigquery_dataset" "silver" {
  dataset_id = "silver_${var.env}"
  location   = "US"
}

resource "google_bigquery_dataset" "gold" {
  dataset_id = "gold_${var.env}"
  location   = "US"
}

# Service account for pipeline
resource "google_service_account" "pipeline" {
  account_id   = "va-pipeline-sa-${var.env}"
  display_name = "VA Analytics Pipeline SA (${var.env})"
}

resource "google_project_iam_member" "pipeline_bq" {
  project = var.project_id
  role    = "roles/bigquery.dataEditor"
  member  = "serviceAccount:${google_service_account.pipeline.email}"
}

# Pub/Sub topic for GCS event notifications
resource "google_pubsub_topic" "new_files" {
  name = "va-new-files-${var.env}"
}

# outputs.tf
output "bronze_bucket" {
  value = google_storage_bucket.bronze.name
}

output "pipeline_sa_email" {
  value = google_service_account.pipeline.email
}
```

---

### Our Pipeline as Terraform HCL — AWS

```hcl
provider "aws" {
  region = var.aws_region
}

resource "aws_s3_bucket" "bronze" {
  bucket = "${var.project_name}-bronze-${var.env}"
}

resource "aws_s3_bucket" "silver" {
  bucket = "${var.project_name}-silver-${var.env}"
}

resource "aws_s3_bucket" "gold" {
  bucket = "${var.project_name}-gold-${var.env}"
}

resource "aws_iam_role" "emr_service" {
  name = "va-emr-service-${var.env}"
  assume_role_policy = jsonencode({
    Version = "2012-10-17"
    Statement = [{
      Effect    = "Allow"
      Principal = { Service = "elasticmapreduce.amazonaws.com" }
      Action    = "sts:AssumeRole"
    }]
  })
}

resource "aws_emr_cluster" "pipeline" {
  name          = "va-pipeline-${var.env}"
  release_label = "emr-7.0.0"
  applications  = ["Spark", "Hadoop"]

  master_instance_group {
    instance_type = "m5.xlarge"
  }
  core_instance_group {
    instance_type  = "m5.xlarge"
    instance_count = 2
  }
}

resource "aws_redshift_cluster" "gold" {
  cluster_identifier = "va-gold-${var.env}"
  database_name      = "va_analytics"
  master_username    = var.redshift_user
  master_password    = var.redshift_password
  node_type          = "dc2.large"
  cluster_type       = "single-node"
  skip_final_snapshot = true
}
```

---

### Why Does This Work?

When you run `terraform apply`:
1. Terraform reads all `.tf` files in the directory
2. Builds a **dependency graph** (e.g., IAM (Identity and Access Management) binding depends on service account existing)
3. Calls GCP/AWS APIs in dependency order
4. Writes what it created to a **state file** (`terraform.tfstate`)
5. Next `terraform plan` compares your `.tf` files to the state file → shows only the delta

---

### Edge Cases / Gotchas

| Scenario | What Happens | How to Handle |
|---|---|---|
| **State file conflict** | Two engineers run apply at the same time | Use remote state (GCS bucket / S3 bucket) with locking |
| **Drift** | Someone changes a resource in the console | `terraform plan` shows the drift; `terraform apply` fixes it |
| **Import existing resource** | You created a bucket manually and want Terraform to manage it | `terraform import google_storage_bucket.bronze my-bucket-name` |
| **Destroy protection** | Someone runs `terraform destroy` on prod | Add `lifecycle { prevent_destroy = true }` to critical resources |

---

### Interview Questions

**Q: What is Terraform state and why does it matter?**
A: State is Terraform's record of what it created. Without it, Terraform can't know what already exists and would try to re-create everything on every apply. State enables incremental updates — only change what's different.

**Q: What's the difference between `terraform plan` and `terraform apply`?**
A: Plan is a dry run — shows what would change without changing anything. Apply executes the plan. Always run plan before apply in production.

**Q: What happens if you delete a resource from your .tf file?**
A: On next `terraform apply`, Terraform destroys that resource. The `.tf` file declares desired state — removing something from desired state means it gets cleaned up.

---
## 4. CDKTF — Hands-On Deep Dive (Python) <a id="4-cdktf"></a>

> **In plain English:** CDKTF (Cloud Development Kit for Terraform) lets you write Python instead of HCL. Under the hood it converts your Python into Terraform JSON, then calls `terraform apply`. You get the full Terraform ecosystem (every cloud provider, state management, plan/apply workflow) while writing code you already know.

**The conversion chain:**

```
Your Python (cdktf) → cdktf synth → Terraform JSON → terraform apply → GCP/AWS API
```

---

### Setup

```bash
# Install CDKTF CLI
npm install -g cdktf-cli

# Install Python provider libraries
pip install cdktf cdktf-cdktf-provider-google

# Initialize a new CDKTF project
mkdir va-infrastructure && cd va-infrastructure
cdktf init --template=python --local

# After init, your project has:
# main.py          ← your infrastructure code
# cdktf.json       ← project config
# requirements.txt ← Python deps
```

---

### Hello World: GCS Bucket in 10 Lines

```python
# main.py
from cdktf import App, TerraformStack, TerraformOutput
from cdktf_cdktf_provider_google.provider import GoogleProvider
from cdktf_cdktf_provider_google.storage_bucket import StorageBucket

class HelloStack(TerraformStack):
    def __init__(self, scope, id):
        super().__init__(scope, id)

        GoogleProvider(self, "google", project="my-project-id", region="us-central1")

        bucket = StorageBucket(self, "hello-bucket",
            name="my-hello-bucket-cdktf",
            location="US",
        )

        TerraformOutput(self, "bucket_name", value=bucket.name)

app = App()
HelloStack(app, "hello")
app.synth()
```

```bash
cdktf synth    # Generates cdktf.out/stacks/hello/cdk.tf.json
cdktf deploy   # Runs terraform apply on the generated JSON
```

**You should see:**
```
hello  Initializing the backend...
hello  Terraform has been successfully initialized!
hello  Plan: 1 to add, 0 to change, 0 to destroy.
hello  Apply complete! Resources: 1 added.
Outputs:
  bucket_name = "my-hello-bucket-cdktf"
```

---

### Why Does This Work?

`cdktf synth` calls `app.synth()`, which traverses the construct tree you built in Python, serializes every resource to Terraform JSON, and writes it to `cdktf.out/`. Then `cdktf deploy` simply runs `terraform apply` on that JSON. The generated JSON is valid Terraform — you can inspect it, check it into Git, and use any Terraform tooling on it.

---

### Full Pipeline Infrastructure in CDKTF

```python
# main.py — Full VA Analytics Pipeline Infrastructure
import os
from constructs import Construct
from cdktf import App, TerraformStack, TerraformOutput
from cdktf_cdktf_provider_google.provider import GoogleProvider
from cdktf_cdktf_provider_google.storage_bucket import StorageBucket
from cdktf_cdktf_provider_google.storage_bucket_iam_member import StorageBucketIamMember
from cdktf_cdktf_provider_google.dataproc_cluster import (
    DataprocCluster, DataprocClusterClusterConfig,
    DataprocClusterClusterConfigMasterConfig,
    DataprocClusterClusterConfigWorkerConfig,
    DataprocClusterClusterConfigPreemptibleWorkerConfig,
)
from cdktf_cdktf_provider_google.bigquery_dataset import BigqueryDataset
from cdktf_cdktf_provider_google.service_account import ServiceAccount
from cdktf_cdktf_provider_google.project_iam_member import ProjectIamMember
from cdktf_cdktf_provider_google.pubsub_topic import PubsubTopic
from cdktf_cdktf_provider_google.storage_notification import StorageNotification


class VAPipelineStack(TerraformStack):
    def __init__(self, scope: Construct, id: str, env: str, project_id: str):
        super().__init__(scope, id)

        region = "us-central1"
        is_prod = (env == "prod")

        # ── Provider ─────────────────────────────────────────────────────
        GoogleProvider(self, "google", project=project_id, region=region)

        # ── Service Account ───────────────────────────────────────────────
        sa = ServiceAccount(self, "pipeline-sa",
            account_id=f"va-pipeline-sa-{env}",
            display_name=f"VA Analytics Pipeline SA ({env})",
        )

        # ── GCS Buckets: Bronze / Silver / Gold ───────────────────────────
        bronze = StorageBucket(self, "bronze",
            name=f"{project_id}-bronze-{env}",
            location="US",
            force_destroy=not is_prod,
            lifecycle_rule=[{
                "condition": {"age": 90},
                "action": {"type": "Delete"},
            }] if not is_prod else [],
        )

        silver = StorageBucket(self, "silver",
            name=f"{project_id}-silver-{env}",
            location="US",
            force_destroy=not is_prod,
        )

        gold = StorageBucket(self, "gold",
            name=f"{project_id}-gold-{env}",
            location="US",
            force_destroy=not is_prod,
        )

        # IAM: allow pipeline SA to write to all buckets
        for bucket, role_id in [(bronze, "bronze"), (silver, "silver"), (gold, "gold")]:
            StorageBucketIamMember(self, f"sa-{role_id}-write",
                bucket=bucket.name,
                role="roles/storage.objectAdmin",
                member=f"serviceAccount:{sa.email}",
            )

        # ── BigQuery Datasets ─────────────────────────────────────────────
        for layer in ["bronze", "silver", "gold"]:
            BigqueryDataset(self, f"bq-{layer}",
                dataset_id=f"{layer}_{env}",
                location="US",
            )

        ProjectIamMember(self, "sa-bq-editor",
            project=project_id,
            role="roles/bigquery.dataEditor",
            member=f"serviceAccount:{sa.email}",
        )

        # ── Dataproc Cluster ──────────────────────────────────────────────
        DataprocCluster(self, "dataproc",
            name=f"va-pipeline-{env}",
            region=region,
            cluster_config=DataprocClusterClusterConfig(
                master_config=DataprocClusterClusterConfigMasterConfig(
                    num_instances=1,
                    machine_type="n1-standard-4",
                ),
                worker_config=DataprocClusterClusterConfigWorkerConfig(
                    num_instances=2,
                    machine_type="n1-standard-4",
                ),
                preemptible_worker_config=DataprocClusterClusterConfigPreemptibleWorkerConfig(
                    num_instances=4 if is_prod else 0,
                ),
            ),
        )

        # ── Pub/Sub Topic + GCS Notification ─────────────────────────────
        topic = PubsubTopic(self, "new-files-topic",
            name=f"va-new-files-{env}",
        )

        StorageNotification(self, "bronze-notification",
            bucket=bronze.name,
            payload_format="JSON_API_V1",
            topic=topic.id,
            event_types=["OBJECT_FINALIZE"],
            object_name_prefix="raw/",
        )

        # ── Outputs ───────────────────────────────────────────────────────
        TerraformOutput(self, "bronze_bucket", value=bronze.name)
        TerraformOutput(self, "silver_bucket", value=silver.name)
        TerraformOutput(self, "gold_bucket",   value=gold.name)
        TerraformOutput(self, "pipeline_sa",   value=sa.email)
        TerraformOutput(self, "pubsub_topic",  value=topic.name)


# ── Entrypoint: one stack per environment ────────────────────────────────────
app = App()
project_id = os.environ.get("GCP_PROJECT_ID", "my-project-id")
VAPipelineStack(app, "va-pipeline-dev",     env="dev",     project_id=project_id)
VAPipelineStack(app, "va-pipeline-staging", env="staging", project_id=project_id)
VAPipelineStack(app, "va-pipeline-prod",    env="prod",    project_id=project_id)
app.synth()
```

```bash
# Deploy dev environment only
cdktf deploy va-pipeline-dev

# Deploy all environments
cdktf deploy '*'

# Preview what prod would change
cdktf diff va-pipeline-prod
```

---

### Inspect the Generated Terraform JSON

```bash
cat cdktf.out/stacks/va-pipeline-dev/cdk.tf.json | python3 -m json.tool | head -60
```

**You should see** a valid Terraform JSON file with `resource`, `provider`, and `output` blocks — identical to what you'd write by hand in HCL, but generated from your Python.

---

### CDKTF vs Raw Terraform Trade-offs

| | CDKTF (Python) | Terraform HCL |
|---|---|---|
| **Language** | Python you already know | HCL (new syntax to learn) |
| **Loops / conditions** | Native Python `for`, `if` | HCL `count`, `for_each` (awkward) |
| **IDE support** | Full autocomplete, type hints | Limited |
| **Ecosystem** | Same Terraform providers | Same Terraform providers |
| **State management** | Same as Terraform | Same as Terraform |
| **Industry adoption** | Growing | Dominant today |
| **When to use** | Python-first teams, complex logic | Existing HCL codebase, pure IaC (Infrastructure as Code) teams |

---

### Edge Cases

- **Provider version pinning:** Always pin provider versions (`~> 5.0`) to avoid breaking changes on `cdktf get`
- **Cross-stack references:** CDKTF supports passing outputs between stacks — useful when your network stack creates a VPC (Virtual Private Cloud) that your cluster stack needs
- **Remote state:** For production, configure a GCS backend for state (`TerraformBackend`) so multiple engineers don't clobber each other

In [ ]:
# ============================================================
# SECTION 4 -- CDKTF: Simulating the Python -> Terraform JSON conversion
# CDKTF = CDK for Terraform
# CDK  = Cloud Development Kit (write cloud infrastructure in a real language)
# IaC  = Infrastructure as Code (define servers/buckets/roles in code, not UI)
# ============================================================
# WHY simulate instead of running the real tool?
# cdktf requires Node.js and terraform binaries -- not available in Colab.
# This cell shows EXACTLY what `cdktf synth` produces so you know what to
# expect when you run it locally or in a CI pipeline.
# You can paste the output JSON into any Terraform workspace and run:
#   terraform init && terraform apply

import json

# ─────────────────────────────────────────────────────────────────────────────
# HELLO-WORLD RAMP: what does one minimal Terraform resource look like in JSON?
# HCL = HashiCorp Configuration Language (the native Terraform file format).
# CDKTF converts your Python class into Terraform JSON instead of HCL.
# ─────────────────────────────────────────────────────────────────────────────
# In HCL you would write:
#   resource "google_storage_bucket" "bronze" {
#     name     = "my-project-bronze-dev"
#     location = "US"
#   }
# CDKTF generates the equivalent JSON automatically from a Python class.
# That JSON is what terraform applies against the Google Cloud API.
hello_world_tf_json = {
    "resource": {
        "google_storage_bucket": {
            "bronze": {"name": "my-project-bronze-dev", "location": "US"}
        }
    }
}
# You should see: minimal Terraform JSON -- a bucket with name and location only.
print("Hello-world Terraform JSON (what cdktf synth produces at minimum):")
print(json.dumps(hello_world_tf_json, indent=2))
print()

# ─────────────────────────────────────────────────────────────────────────────
# Full simulation function -- builds the complete Terraform JSON that
# cdktf synth would emit for the VA Analytics pipeline stack.
# ─────────────────────────────────────────────────────────────────────────────
def simulate_cdktf_synth(project_id: str, env: str) -> dict:
    """
    Simulates what `cdktf synth` produces for the bronze GCS bucket.
    The real cdktf library generates this automatically from your Python class.
    GCS = Google Cloud Storage (equivalent to AWS S3)
    SA  = Service Account (a non-human identity used by pipelines to access GCP)
    """
    # WHY embed env in resource names?
    # Separate name-spaced resources per environment prevent accidental
    # cross-environment reads/writes (dev code touching prod data).
    bucket_name   = f"{project_id}-bronze-{env}"
    sa_account_id = f"va-pipeline-sa-{env}"

    terraform_json = {
        # terraform block tells the provider which plugin version to download
        "terraform": {
            "required_providers": {
                "google": {
                    "source": "hashicorp/google",
                    "version": "~> 5.0"   # ~> means compatible with 5.x
                }
            }
        },
        # provider block sets the default GCP project and region
        "provider": {
            "google": [{
                "project": project_id,
                "region": "us-central1"
            }]
        },
        "resource": {
            # GCS (Google Cloud Storage) bucket for raw/Bronze call-center files
            "google_storage_bucket": {
                "bronze": {
                    "name": bucket_name,
                    "location": "US",
                    # WHY force_destroy only in non-prod?
                    # In dev/staging you want `terraform destroy` to clean up
                    # everything without manual intervention. In prod you never
                    # want an accidental destroy to delete customer data.
                    "force_destroy": env != "prod",
                    # WHY a lifecycle rule in non-prod but not prod?
                    # Dev data expires -- keeping it forever wastes money.
                    # Prod data is retained indefinitely for audit/compliance.
                    "lifecycle_rule": [
                        {"condition": {"age": 90}, "action": {"type": "Delete"}}
                    ] if env != "prod" else []
                }
            },
            # SA = Service Account used by Dataproc and Cloud Composer
            "google_service_account": {
                "pipeline-sa": {
                    "account_id": sa_account_id,
                    "display_name": f"VA Analytics Pipeline SA ({env})"
                }
            },
            # IAM = Identity and Access Management (GCP permissions system)
            # This binding lets the SA write objects to the Bronze bucket.
            "google_storage_bucket_iam_member": {
                "sa-bronze-write": {
                    # ${...} is Terraform interpolation syntax -- resolved at apply time
                    "bucket": f"${{google_storage_bucket.bronze.name}}",
                    "role": "roles/storage.objectAdmin",
                    "member": f"serviceAccount:${{google_service_account.pipeline-sa.email}}"
                }
            }
        },
        # Outputs are printed after `terraform apply` and saved in state
        "output": {
            "bronze_bucket": {"value": "${google_storage_bucket.bronze.name}"},
            "pipeline_sa":   {"value": "${google_service_account.pipeline-sa.email}"}
        }
    }
    return terraform_json

# ─────────────────────────────────────────────────────────────────────────────
# Generate and display the synth output for dev and prod environments
# ─────────────────────────────────────────────────────────────────────────────
for env in ["dev", "prod"]:
    tf_json = simulate_cdktf_synth("quintrix-va-analytics", env)
    print(f"=== cdktf synth output for env={env} ===")
    # Truncate at 1200 chars to keep the notebook output readable
    print(json.dumps(tf_json, indent=2)[:1200])
    print("  ... (truncated)")
    print()

# ─────────────────────────────────────────────────────────────────────────────
# Key difference between dev and prod stacks
# WHY compare explicitly? This is the core value of CDKTF -- a single Python
# conditional controls infra behaviour across environments, not copy-pasted HCL.
# ─────────────────────────────────────────────────────────────────────────────
dev  = simulate_cdktf_synth("quintrix-va-analytics", "dev")
prod = simulate_cdktf_synth("quintrix-va-analytics", "prod")

# You should see: force_destroy=True for dev, False for prod
print("Key differences between dev and prod stacks:")
print(f"  dev  force_destroy  = {dev['resource']['google_storage_bucket']['bronze']['force_destroy']}")
print(f"  prod force_destroy  = {prod['resource']['google_storage_bucket']['bronze']['force_destroy']}")
print(f"  dev  lifecycle_rule = {len(dev['resource']['google_storage_bucket']['bronze']['lifecycle_rule'])} rule(s)")
print(f"  prod lifecycle_rule = {len(prod['resource']['google_storage_bucket']['bronze']['lifecycle_rule'])} rule(s)")
print()
print("This is the power of CDKTF: Python conditionals control infra.")
# HCL = HashiCorp Configuration Language (Terraform native config format)
print("In raw HCL you would use count = var.env != 'prod' ? 1 : 0 -- much harder to read.")

---
## 5. AWS CDK — Same Pipeline on AWS <a id="5-aws-cdk"></a>

> **In plain English:** AWS CDK does for AWS what CDKTF does for GCP — lets you write Python instead of clicking the console. But the output is different: instead of Terraform JSON, CDK generates CloudFormation YAML. CloudFormation is AWS's native IaC service. You write Python, CDK converts it to CloudFormation, CloudFormation talks to AWS.

**The conversion chain:**

```
Your Python (aws-cdk-lib) → cdk synth → CloudFormation YAML → AWS CloudFormation → AWS APIs
```

---

### Setup

```bash
# Install AWS CDK CLI
npm install -g aws-cdk

# Install Python library
pip install aws-cdk-lib constructs

# Initialize project
mkdir va-aws-infrastructure && cd va-aws-infrastructure
cdk init app --language=python
```

---

### Full Pipeline Infrastructure in AWS CDK

```python
# va_aws_stack.py
import os
import aws_cdk as cdk
from aws_cdk import (
    Stack,
    aws_s3 as s3,
    aws_iam as iam,
    aws_emr as emr,
    aws_redshift as redshift,
    aws_sqs as sqs,
    aws_s3_notifications as s3n,
    aws_lambda as lambda_,
    CfnOutput,
)
from constructs import Construct


class VAAWSStack(Stack):
    def __init__(self, scope: Construct, id: str, env_name: str, **kwargs):
        super().__init__(scope, id, **kwargs)

        is_prod = (env_name == "prod")

        # ── S3 Buckets ────────────────────────────────────────────────────
        bronze = s3.Bucket(self, "Bronze",
            bucket_name=f"quintrix-va-bronze-{env_name}",
            removal_policy=cdk.RemovalPolicy.DESTROY if not is_prod else cdk.RemovalPolicy.RETAIN,
            auto_delete_objects=not is_prod,
            lifecycle_rules=[
                s3.LifecycleRule(expiration=cdk.Duration.days(90))
            ] if not is_prod else [],
        )

        silver = s3.Bucket(self, "Silver",
            bucket_name=f"quintrix-va-silver-{env_name}",
            removal_policy=cdk.RemovalPolicy.DESTROY if not is_prod else cdk.RemovalPolicy.RETAIN,
        )

        gold = s3.Bucket(self, "Gold",
            bucket_name=f"quintrix-va-gold-{env_name}",
            removal_policy=cdk.RemovalPolicy.DESTROY if not is_prod else cdk.RemovalPolicy.RETAIN,
        )

        # ── IAM Role for EMR ──────────────────────────────────────────────
        emr_role = iam.Role(self, "EMRServiceRole",
            role_name=f"va-emr-service-{env_name}",
            assumed_by=iam.ServicePrincipal("elasticmapreduce.amazonaws.com"),
            managed_policies=[
                iam.ManagedPolicy.from_aws_managed_policy_name("service-role/AmazonElasticMapReduceRole")
            ],
        )

        emr_ec2_role = iam.Role(self, "EMREC2Role",
            role_name=f"va-emr-ec2-{env_name}",
            assumed_by=iam.ServicePrincipal("ec2.amazonaws.com"),
            managed_policies=[
                iam.ManagedPolicy.from_aws_managed_policy_name("service-role/AmazonElasticMapReduceforEC2Role")
            ],
        )
        bronze.grant_read_write(emr_ec2_role)
        silver.grant_read_write(emr_ec2_role)
        gold.grant_read_write(emr_ec2_role)

        # ── EMR Cluster ───────────────────────────────────────────────────
        # Note: aws-cdk-lib uses CfnCluster for EMR (L1 construct)
        emr_cluster = emr.CfnCluster(self, "EMRCluster",
            name=f"va-pipeline-{env_name}",
            release_label="emr-7.0.0",
            applications=[
                emr.CfnCluster.ApplicationProperty(name="Spark"),
                emr.CfnCluster.ApplicationProperty(name="Hadoop"),
            ],
            instances=emr.CfnCluster.JobFlowInstancesConfigProperty(
                master_instance_group=emr.CfnCluster.InstanceGroupConfigProperty(
                    instance_count=1,
                    instance_type="m5.xlarge",
                ),
                core_instance_group=emr.CfnCluster.InstanceGroupConfigProperty(
                    instance_count=2 if is_prod else 1,
                    instance_type="m5.xlarge",
                ),
            ),
            job_flow_role=emr_ec2_role.role_name,
            service_role=emr_role.role_name,
        )

        # ── SQS (Simple Queue Service) Queue + S3 Event Notification ────────────────────────────
        new_files_queue = sqs.Queue(self, "NewFilesQueue",
            queue_name=f"va-new-files-{env_name}",
            visibility_timeout=cdk.Duration.seconds(300),
        )

        bronze.add_event_notification(
            s3.EventType.OBJECT_CREATED,
            s3n.SqsDestination(new_files_queue),
            s3.NotificationKeyFilter(prefix="raw/"),
        )

        # ── Outputs ───────────────────────────────────────────────────────
        CfnOutput(self, "BronzeBucket", value=bronze.bucket_name)
        CfnOutput(self, "SilverBucket", value=silver.bucket_name)
        CfnOutput(self, "GoldBucket",   value=gold.bucket_name)
        CfnOutput(self, "EMRClusterId", value=emr_cluster.ref)
        CfnOutput(self, "NewFilesQueue", value=new_files_queue.queue_url)


# app.py
app = cdk.App()
env_name = os.environ.get("DEPLOY_ENV", "dev")
VAAWSStack(app, f"va-aws-{env_name}",
    env_name=env_name,
    env=cdk.Environment(
        account=os.environ.get("CDK_DEFAULT_ACCOUNT"),
        region=os.environ.get("CDK_DEFAULT_REGION", "us-east-1"),
    ),
)
app.synth()
```

```bash
# Preview (generates CloudFormation YAML to cdk.out/)
cdk synth

# Deploy
cdk deploy va-aws-dev

# Show CloudFormation template
cat cdk.out/va-aws-dev.template.json | python3 -m json.tool | head -80
```

---

### CDKTF vs AWS CDK — Side-by-Side

| | CDKTF | AWS CDK |
|---|---|---|
| **Output format** | Terraform JSON | CloudFormation YAML/JSON |
| **Cloud support** | Multi-cloud | AWS only |
| **State management** | Terraform state file | CloudFormation manages state internally |
| **Rollback** | `terraform apply` previous version | CloudFormation automatic rollback on failure |
| **Provider ecosystem** | All Terraform providers (1,000+) | AWS services only (via CDK constructs) |
| **Python experience** | Similar | Similar |
| **Best for** | GCP-primary or multi-cloud | AWS-native teams |

**For our cohort:** CDKTF for GCP work. AWS CDK when the client is AWS-only.

In [ ]:
# ============================================================
# SECTION 5 -- AWS CDK: Simulating the CloudFormation output
# CDK           = Cloud Development Kit (AWS IaC tool)
# IaC           = Infrastructure as Code
# CloudFormation = AWS infrastructure provisioning engine
# ============================================================
# WHY simulate?
# `cdk synth` requires the AWS CDK CLI and language runtime, unavailable
# in Colab/Jupyter. This function builds the CloudFormation JSON that CDK
# would produce, so you can see what AWS actually receives.
#
# Real workflow:
#   cdk synth            -> generates cdk.out/<StackName>.template.json
#   cdk deploy --env dev -> uploads that JSON to CloudFormation and applies it

import json

# ─────────────────────────────────────────────────────────────────────────────
# HELLO-WORLD RAMP: the smallest possible CloudFormation template
# CloudFormation uses JSON or YAML with a fixed structure:
#   AWSTemplateFormatVersion, Resources (required), and optional Outputs.
# ─────────────────────────────────────────────────────────────────────────────
hello_world_cfn = {
    "AWSTemplateFormatVersion": "2010-09-09",
    "Resources": {
        "MyBucket": {
            "Type": "AWS::S3::Bucket",
            "Properties": {"BucketName": "my-hello-world-bucket-dev"}
        }
    }
}
# You should see: a minimal CloudFormation template with one S3 bucket.
print("Hello-world CloudFormation template (what cdk synth produces at minimum):")
print(json.dumps(hello_world_cfn, indent=2))
print()

# ─────────────────────────────────────────────────────────────────────────────
# Full simulation for the VA Analytics pipeline AWS stack
# ─────────────────────────────────────────────────────────────────────────────
def simulate_cdk_synth(env_name: str) -> dict:
    """
    Simulates a subset of the CloudFormation template that cdk synth generates.
    In real usage: cat cdk.out/va-aws-dev.template.json
    S3  = Simple Storage Service (AWS object storage)
    SQS = Simple Queue Service (AWS message queue)
    IAM = Identity and Access Management (AWS permissions system)
    EMR = Elastic MapReduce (AWS managed Spark/Hadoop cluster service)
    """
    is_prod = (env_name == "prod")
    # WHY two different removal policies?
    # Retain keeps the S3 bucket when the CloudFormation stack is deleted.
    # Delete removes it. You never want prod data accidentally deleted,
    # but dev environments should clean up after themselves.
    removal_policy = "Retain" if is_prod else "Delete"

    template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Description": f"VA Analytics Pipeline -- {env_name}",
        "Resources": {
            # Storage layer: Bronze / Silver / Gold S3 buckets
            # These map to the medallion architecture tiers.
            "Bronze": {
                "Type": "AWS::S3::Bucket",
                "Properties": {
                    "BucketName": f"quintrix-va-bronze-{env_name}",
                },
                # DeletionPolicy: what happens when the stack is deleted
                "DeletionPolicy": removal_policy,
                # UpdateReplacePolicy: what happens when the resource must be
                # replaced (e.g., bucket name changed -- S3 requires delete+recreate)
                "UpdateReplacePolicy": removal_policy,
            },
            "Silver": {
                "Type": "AWS::S3::Bucket",
                "Properties": {
                    "BucketName": f"quintrix-va-silver-{env_name}",
                },
                "DeletionPolicy": removal_policy,
            },
            "Gold": {
                "Type": "AWS::S3::Bucket",
                "Properties": {
                    "BucketName": f"quintrix-va-gold-{env_name}",
                },
                "DeletionPolicy": removal_policy,
            },
            # Event queue: triggers Spark job when a new Bronze file lands
            # SQS = Simple Queue Service -- decouples ingest trigger from
            # the processing job so backpressure is handled automatically.
            "NewFilesQueue": {
                "Type": "AWS::SQS::Queue",
                "Properties": {
                    "QueueName": f"va-new-files-{env_name}",
                    # VisibilityTimeout: how long a consumer has to process a
                    # message before SQS makes it visible to other consumers.
                    # 300s = 5 minutes -- enough for a short Spark job.
                    "VisibilityTimeout": 300,
                }
            },
            # IAM role for EMR (Elastic MapReduce) cluster
            # EMR needs permission to call AWS services on your behalf.
            "EMRServiceRole": {
                "Type": "AWS::IAM::Role",
                "Properties": {
                    "RoleName": f"va-emr-service-{env_name}",
                    "AssumeRolePolicyDocument": {
                        "Version": "2012-10-17",
                        "Statement": [{
                            "Effect": "Allow",
                            # elasticmapreduce.amazonaws.com is the service
                            # principal that EMR uses to request role credentials
                            "Principal": {"Service": "elasticmapreduce.amazonaws.com"},
                            # sts:AssumeRole = permission that lets EMR act as
                            # this role and call AWS APIs on your behalf
                            "Action": "sts:AssumeRole"
                        }]
                    }
                }
            }
        },
        # Outputs are printed after `cdk deploy` and visible in the AWS console
        "Outputs": {
            "BronzeBucket":  {"Value": {"Ref": "Bronze"}},
            "SilverBucket":  {"Value": {"Ref": "Silver"}},
            "GoldBucket":    {"Value": {"Ref": "Gold"}},
            "NewFilesQueue": {"Value": {"Ref": "NewFilesQueue"}},
        }
    }
    return template

# ─────────────────────────────────────────────────────────────────────────────
# Compare dev vs prod Bronze bucket config -- DeletionPolicy is the key diff
# ─────────────────────────────────────────────────────────────────────────────
print("=== CloudFormation template snippet (dev) ===")
dev_template  = simulate_cdk_synth("dev")
print(json.dumps(dev_template["Resources"]["Bronze"], indent=2))

print()
print("=== CloudFormation template snippet (prod) ===")
prod_template = simulate_cdk_synth("prod")
print(json.dumps(prod_template["Resources"]["Bronze"], indent=2))

# You should see: DeletionPolicy=Delete for dev, Retain for prod.
print()
print("Key insight: CDK output = CloudFormation JSON/YAML.")
print("  CDKTF output = Terraform JSON.")
print("  Both start as Python. Different IaC engines at the end.")
print()
# Counting resources shows the full scope of what one cdk deploy provisions
print(f"Total resources in dev template:  {len(dev_template['Resources'])}")
print(f"Total resources in prod template: {len(prod_template['Resources'])}")

---
## 6. GitHub Actions — CI Pipeline <a id="6-github-actions"></a>

> **In plain English:** GitHub Actions is a robot that reviews your homework before the teacher sees it. You submit a pull request (homework). The robot checks spelling (linting), checks the math (unit tests), validates that your Airflow DAG actually imports without errors, and previews what infrastructure would change. If anything fails, the PR is blocked. The teacher (reviewer) only sees it after the robot approves.

---

### How GitHub Actions Works

```
Developer pushes branch → opens Pull Request
  ↓
GitHub detects .github/workflows/ci.yml
  ↓
GitHub spins up a fresh Ubuntu VM (runner)
  ↓
Runner executes your steps in order
  ↓
Each step reports pass ✓ or fail ✗
  ↓
All pass → green checkmark on PR → reviewer can merge
One fails → red X → PR blocked → developer fixes and pushes again
```

---

### Hello World: Minimal CI Workflow

```yaml
# .github/workflows/ci.yml
name: CI

on:
  pull_request:
    branches: [ main ]

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install ruff
      - run: ruff check src/
```

That's it. Every PR to `main` now runs a linter. 12 lines of YAML.

---

### Full CI Pipeline for Our Data Project

```yaml
# .github/workflows/ci.yml
name: Data Pipeline CI

on:
  pull_request:
    branches: [ main ]
  push:
    branches: [ main ]

env:
  PYTHON_VERSION: "3.11"

jobs:
  # ─── Job 1: Lint and Test ─────────────────────────────────────────────────
  lint-and-test:
    name: Lint + Unit Tests
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install ruff pytest pytest-cov

      - name: Lint PySpark scripts (ruff)
        run: |
          ruff check src/bronze_ingest.py src/silver_transform.py
          echo "Lint passed"

      - name: Run unit tests
        run: |
          pytest tests/ -v --cov=src --cov-report=term-missing
          echo "All tests passed"

      - name: Upload coverage report
        uses: actions/upload-artifact@v4
        with:
          name: coverage-report
          path: .coverage

  # ─── Job 2: Validate Airflow DAGs ────────────────────────────────────────
  validate-dags:
    name: Validate Airflow DAGs
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - name: Install Airflow
        run: |
          pip install apache-airflow==2.9.0 --constraint             "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.0/constraints-3.11.txt"

      - name: Validate DAG imports
        run: |
          python -c "
          import sys, importlib
          sys.path.insert(0, 'dags/')
          mod = importlib.import_module('va_analytics_pipeline')
          assert hasattr(mod, 'dag'), 'No dag object found in DAG file'
          print('DAG import OK:', mod.dag.dag_id)
          "

      - name: List DAGs (airflow parse)
        run: |
          export AIRFLOW__CORE__DAGS_FOLDER=dags/
          export AIRFLOW__DATABASE__SQL_ALCHEMY_CONN=sqlite:///airflow.db
          airflow db init
          airflow dags list
          airflow dags list-import-errors

  # ─── Job 3: Infrastructure Plan ──────────────────────────────────────────
  infra-plan:
    name: CDKTF Plan (GCP)
    runs-on: ubuntu-latest
    if: github.event_name == 'pull_request'

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-node@v4
        with:
          node-version: "20"
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install CDKTF
        run: npm install -g cdktf-cli

      - name: Install Python deps
        run: pip install cdktf cdktf-cdktf-provider-google

      - name: CDKTF synth (validate Python → Terraform JSON)
        run: cdktf synth
        env:
          GCP_PROJECT_ID: ${{ secrets.GCP_PROJECT_ID }}

      - name: Terraform plan (show infra diff — no apply)
        run: cdktf diff va-pipeline-staging
        env:
          GCP_PROJECT_ID: ${{ secrets.GCP_PROJECT_ID }}
          GOOGLE_CREDENTIALS: ${{ secrets.GCP_SA_KEY }}

  # ─── Job 4: Data Quality Dry Run ─────────────────────────────────────────
  data-quality:
    name: Data Quality Validation
    runs-on: ubuntu-latest
    needs: lint-and-test

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - name: Install quality deps
        run: pip install great_expectations pandas

      - name: Run quality rules (dry run on sample data)
        run: python scripts/run_quality_checks.py --dry-run
```

---

### Secrets in CI

Never put credentials in your YAML file. Use **GitHub Secrets**:

```
GitHub → Your Repo → Settings → Secrets and variables → Actions
```

| Secret Name | Value |
|---|---|
| `GCP_PROJECT_ID` | `quintrix-va-analytics` |
| `GCP_SA_KEY` | Service account JSON key (base64 encoded) |
| `AWS_ACCESS_KEY_ID` | AWS access key |
| `AWS_SECRET_ACCESS_KEY` | AWS secret key |

In your workflow: `${{ secrets.GCP_SA_KEY }}` — GitHub replaces this at runtime and masks it in logs.

---

### Why Does This Work?

GitHub maintains a fleet of VMs (called **runners**). When your workflow triggers, GitHub:
1. Picks an available `ubuntu-latest` VM
2. Clones your repo into it
3. Runs your steps in order — each step in the same shell session
4. Reports pass/fail back to the PR

The VM is ephemeral — destroyed after the job. Every run starts clean. This is why you `pip install` in every run (or cache with `cache: pip`).

---

### Console Walkthrough

```
Your Repo → Actions tab → click a workflow run → click a job → expand steps
```

You can:
- View logs in real time (stream)
- Download artifacts (coverage reports, test results)
- Re-run failed jobs with one click
- Cancel a running job

---

### Edge Cases / Gotchas

| Scenario | Problem | Fix |
|---|---|---|
| **Slow installs** | `pip install apache-airflow` takes 3 minutes | Use `cache: pip` in `setup-python` step |
| **Matrix testing** | Need to test on Python 3.10 and 3.11 | Use `strategy: matrix: python-version: [3.10, 3.11]` |
| **Flaky tests** | Test passes locally, fails in CI | Check for hardcoded paths, missing env vars, timezone differences |
| **Secret rotation** | SA key expires, CI fails | Use Workload Identity Federation instead of static keys |
| **Parallel jobs** | `infra-plan` takes 5 min, blocking merge | Run it in parallel with `lint-and-test` (no `needs:` dependency) |

---

### Interview Questions

**Q: What is the difference between a GitHub Actions job and a step?**
A: A job is a unit of work that runs on one runner VM. Steps are sequential commands within a job. Jobs run in parallel by default; use `needs:` to create dependencies between jobs.

**Q: Why do CI pipelines use fresh VMs instead of reusing the same machine?**
A: Isolation. If one run leaves behind stale files, cached state, or modified environment variables, the next run might behave differently. Fresh VMs guarantee reproducibility.

**Q: How do you pass a value from one step to the next in GitHub Actions?**
A: Use step outputs: `echo "my_value=foo" >> $GITHUB_OUTPUT` in one step, then `${{ steps.step_id.outputs.my_value }}` in the next.

In [ ]:
# ============================================================
# SECTION 6 -- GitHub Actions: Simulating the CI pipeline locally
# CI     = Continuous Integration (automated checks on every PR push)
# PR     = Pull Request (a proposed code change awaiting review and CI checks)
# pytest = Python standard unit-testing framework
# ============================================================
# WHY run tests in a notebook?
# GitHub Actions runs these same functions with `pytest tests/` in a container.
# Understanding what the tests do -- and why they FAIL -- is the skill.
# Running them here first lets you see the output before wiring up the YAML.

import json
from datetime import date, datetime

# ─────────────────────────────────────────────────────────────────────────────
# HELLO-WORLD RAMP: the smallest possible pytest test
# A pytest test is just a function whose name starts with test_.
# pytest discovers and runs all such functions automatically.
# ─────────────────────────────────────────────────────────────────────────────
def test_hello_world():
    """Smallest possible test -- proves the test runner works."""
    result = 1 + 1
    # assert = raise AssertionError if the condition is False.
    # pytest intercepts AssertionError and marks the test as FAILED.
    assert result == 2, f"Expected 2, got {result}"
    print("PASS test_hello_world")

# Run the hello-world test immediately so you see its output
test_hello_world()
print()

# ─────────────────────────────────────────────────────────────────────────────
# The production function under test (in real life: src/silver_transform.py)
# Silver transform = medallion-layer transform that cleans Bronze call records
# ─────────────────────────────────────────────────────────────────────────────
def silver_transform(calls: list) -> list:
    """
    Silver transform: clean and standardize call center data.
    - Adds duration_min (converted from raw seconds)
    - Converts outcome string to converted flag (0 or 1)
    - Drops records with null call_date (data quality enforcement)
    """
    result = []
    for row in calls:
        # WHY drop nulls here rather than downstream?
        # Silver is the data quality checkpoint. Letting nulls reach Gold
        # corrupts aggregations that assume every row has a valid date.
        if not row.get("call_date"):
            continue  # drop nulls -- this row will not appear in Silver
        result.append({
            "call_id":      row["call_id"],
            "call_date":    row["call_date"],
            # WHY round to 2 decimal places?
            # Analysts query duration_min in BigQuery; unnecessary precision
            # wastes storage and makes numbers harder to read.
            "duration_min": round(row["duration_sec"] / 60, 2),
            # WHY not store the raw outcome string?
            # Numeric flag enables SUM(converted) in SQL without CASE WHEN.
            "converted":    1 if row.get("outcome") == "converted" else 0,
            "campaign_key": row["campaign_key"],
        })
    return result

# ─────────────────────────────────────────────────────────────────────────────
# Unit tests -- these live in tests/test_silver_transform.py in the real repo.
# Each test function isolates one behaviour so failures are easy to diagnose.
# ─────────────────────────────────────────────────────────────────────────────

def test_basic_transform():
    """Silver transform produces expected columns and correct values."""
    input_data = [
        {"call_id": "C001", "call_date": "2026-03-20",
         "duration_sec": 120, "outcome": "converted", "campaign_key": 1},
    ]
    result = silver_transform(input_data)
    # One row in, one row out (no nulls to drop)
    assert len(result) == 1
    # 120 seconds / 60 = 2.0 minutes
    assert result[0]["duration_min"] == 2.0
    # outcome=converted should map to converted=1
    assert result[0]["converted"] == 1
    # Raw column 'outcome' must NOT appear in Silver -- it has been encoded
    assert "outcome" not in result[0], "Raw column 'outcome' should be dropped"
    print("PASS test_basic_transform")

def test_null_call_date_dropped():
    """Rows with null call_date are dropped in Silver (data quality gate)."""
    input_data = [
        # C001 has null call_date -- should be dropped
        {"call_id": "C001", "call_date": None,
         "duration_sec": 60, "outcome": "converted", "campaign_key": 1},
        # C002 is valid -- should survive the transform
        {"call_id": "C002", "call_date": "2026-03-20",
         "duration_sec": 90, "outcome": "not_converted", "campaign_key": 2},
    ]
    result = silver_transform(input_data)
    # Only C002 should survive -- the f-string makes the failure message useful
    assert len(result) == 1, f"Expected 1 row, got {len(result)}"
    assert result[0]["call_id"] == "C002"
    print("PASS test_null_call_date_dropped")

def test_schema_columns():
    """Output schema matches expected columns -- catches column renames.

    BEGINNER NOTE: This is the most important test in a data pipeline.
    Downstream jobs (Gold mart, dashboards) depend on exact column names.
    Any rename breaks them silently unless this test catches it first.
    """
    EXPECTED_COLUMNS = {"call_id", "call_date", "duration_min", "converted", "campaign_key"}
    input_data = [
        {"call_id": "C001", "call_date": "2026-03-20",
         "duration_sec": 60, "outcome": "converted", "campaign_key": 1},
    ]
    result = silver_transform(input_data)
    actual_columns = set(result[0].keys())
    # WHY compare sets? Sets give you exact Added/Removed diffs, which makes
    # the failure message actionable -- you know exactly what changed.
    assert actual_columns == EXPECTED_COLUMNS, (
        f"Schema mismatch!\n"
        f"  Expected: {sorted(EXPECTED_COLUMNS)}\n"
        f"  Got:      {sorted(actual_columns)}\n"
        f"  Added:    {actual_columns - EXPECTED_COLUMNS}\n"
        f"  Removed:  {EXPECTED_COLUMNS - actual_columns}"
    )
    print("PASS test_schema_columns")

def test_not_converted():
    """Non-converted calls (outcome=callback) produce converted=0."""
    input_data = [
        {"call_id": "C001", "call_date": "2026-03-20",
         "duration_sec": 30, "outcome": "callback", "campaign_key": 3},
    ]
    result = silver_transform(input_data)
    # callback is not converted, so the flag must be 0
    assert result[0]["converted"] == 0
    print("PASS test_not_converted")

def test_empty_input():
    """Empty input list produces empty output without raising an error.

    BEGINNER NOTE: Always test edge cases -- empty input, single row, and
    maximum-size input. Pipelines regularly receive empty files during
    off-hours or when an upstream source is down.
    """
    result = silver_transform([])
    assert result == []
    print("PASS test_empty_input")

# ─────────────────────────────────────────────────────────────────────────────
# Test runner -- mimics what `pytest tests/test_silver_transform.py -v` does
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 55)
print("  Running: pytest tests/test_silver_transform.py -v")
print("=" * 55)
print()

# Collect all test functions in the order pytest would discover them
tests = [
    test_basic_transform,
    test_null_call_date_dropped,
    test_schema_columns,
    test_not_converted,
    test_empty_input,
]

passed = 0
failed = 0
for test_fn in tests:
    try:
        test_fn()
        passed += 1
    except AssertionError as e:
        # pytest prints the AssertionError message to help the developer fix it
        print(f"FAIL {test_fn.__name__}: {e}")
        failed += 1

print()
print(f"Results: {passed} passed, {failed} failed")
print()

if failed == 0:
    # You should see: all 5 PASS, CI step unblocked
    print("GitHub Actions CI step: PASS")
    print("PR is unblocked -- reviewer can approve and merge.")
else:
    print("GitHub Actions CI step: FAIL")
    print("PR is blocked -- developer must fix failing tests before merge.")

# ─────────────────────────────────────────────────────────────────────────────
# Simulate the broken transform -- same tests, but the column was renamed.
# WHY: this shows that test_schema_columns catches renames automatically.
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("  Simulating the broken transform (column rename)")
print("=" * 55)

def silver_transform_broken(calls: list) -> list:
    """Broken version: converted renamed to call_result."""
    result = []
    for row in calls:
        if not row.get("call_date"):
            continue
        result.append({
            "call_id":      row["call_id"],
            "call_date":    row["call_date"],
            "duration_min": round(row["duration_sec"] / 60, 2),
            # BREAKING CHANGE: this renames the column.
            # Gold mart still expects 'converted' -- will silently get 0.
            "call_result":  row.get("outcome"),   # renamed! breaks schema test
            "campaign_key": row["campaign_key"],
        })
    return result

def patched_schema_test():
    """Runs test_schema_columns against the broken transform."""
    EXPECTED_COLUMNS = {"call_id", "call_date", "duration_min", "converted", "campaign_key"}
    input_data = [
        {"call_id": "C001", "call_date": "2026-03-20",
         "duration_sec": 60, "outcome": "converted", "campaign_key": 1},
    ]
    result = silver_transform_broken(input_data)
    actual_columns = set(result[0].keys())
    assert actual_columns == EXPECTED_COLUMNS, (
        f"Schema mismatch! Added: {actual_columns - EXPECTED_COLUMNS}, "
        f"Removed: {EXPECTED_COLUMNS - actual_columns}"
    )

print()
try:
    patched_schema_test()
except AssertionError as e:
    # You should see: Added={'call_result'}, Removed={'converted'}
    print(f"FAIL test_schema_columns: {e}")
    print()
    print("GitHub Actions CI step: FAIL")
    print("PR BLOCKED -- the column rename bug is caught before merge.")
    print("Developer fixes it. Pushes again. CI re-runs. Merge unblocked.")

---
## 7. Environment Promotion: dev → staging → prod <a id="7-environments"></a>

> **In plain English:** A restaurant doesn't test a new recipe on paying customers. The chef tests it in the kitchen (dev). The staff eats it for feedback (staging). Then it goes on the menu (prod). Data pipelines follow the same rule. You don't deploy to production every time you change a line of code.

---

### Three Environments for Our Pipeline

| Environment | Where | Data | Cluster Size | Who Uses It |
|---|---|---|---|---|
| **dev** | Laptop / Cloud Shell | Synthetic sample (1,000 calls) | None (local PySpark) | Developer only |
| **staging** | Full GCP / AWS | Copy of prod, anonymized (last 30 days) | Small (2 workers) | QA, Tech Lead review |
| **prod** | Full GCP / AWS | Real data, real dashboards | Full (4+ workers) | Business, VP, Client |

---

### Environment-Specific Variables

Every part of your pipeline changes by environment. **Never hardcode.**

```python
# config.py — environment-aware configuration
import os

ENV = os.environ.get("DEPLOY_ENV", "dev")

CONFIG = {
    "dev": {
        "gcs_bronze_bucket":  "quintrix-va-bronze-dev",
        "gcs_silver_bucket":  "quintrix-va-silver-dev",
        "gcs_gold_bucket":    "quintrix-va-gold-dev",
        "bq_bronze_dataset":  "bronze_dev",
        "bq_silver_dataset":  "silver_dev",
        "bq_gold_dataset":    "gold_dev",
        "dataproc_cluster":   "va-pipeline-dev",
        "preemptible_workers": 0,
    },
    "staging": {
        "gcs_bronze_bucket":  "quintrix-va-bronze-staging",
        "gcs_silver_bucket":  "quintrix-va-silver-staging",
        "gcs_gold_bucket":    "quintrix-va-gold-staging",
        "bq_bronze_dataset":  "bronze_staging",
        "bq_silver_dataset":  "silver_staging",
        "bq_gold_dataset":    "gold_staging",
        "dataproc_cluster":   "va-pipeline-staging",
        "preemptible_workers": 2,
    },
    "prod": {
        "gcs_bronze_bucket":  "quintrix-va-bronze-prod",
        "gcs_silver_bucket":  "quintrix-va-silver-prod",
        "gcs_gold_bucket":    "quintrix-va-gold-prod",
        "bq_bronze_dataset":  "bronze_prod",
        "bq_silver_dataset":  "silver_prod",
        "bq_gold_dataset":    "gold_prod",
        "dataproc_cluster":   "va-pipeline-prod",
        "preemptible_workers": 4,
    },
}

def get_config():
    return CONFIG[ENV]
```

---

### Branch Strategy for Data Pipelines

```
feature/fix-utc-timezone-bug
  │
  ├── Developer opens PR to main
  │     └── CI runs (lint, test, DAG validate, infra plan)
  │
  ├── PR approved → merge to main
  │     └── CD deploys to STAGING automatically
  │
  ├── QA validates in staging (run pipeline on 30-day sample)
  │     └── Data quality checks pass
  │
  └── Manual approval → CD deploys to PROD
```

---

### CDKTF Stacks as Environments

```python
# In your CDKTF main.py — one stack per environment
app = App()
VAPipelineStack(app, "va-pipeline-dev",     env="dev",     project_id=project_id)
VAPipelineStack(app, "va-pipeline-staging", env="staging", project_id=project_id)
VAPipelineStack(app, "va-pipeline-prod",    env="prod",    project_id=project_id)
app.synth()
```

```bash
# Deploy only staging
cdktf deploy va-pipeline-staging

# Show diff for prod (preview only — don't deploy)
cdktf diff va-pipeline-prod
```

---

### GitHub Actions CD: Staging on Merge, Prod on Manual Approval

```yaml
# .github/workflows/cd.yml
name: CD

on:
  push:
    branches: [ main ]        # staging auto-deploy
  workflow_dispatch:           # prod: manual trigger
    inputs:
      environment:
        type: choice
        options: [ staging, prod ]
        default: staging

jobs:
  deploy-staging:
    name: Deploy to Staging
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main' && github.event_name == 'push'
    environment: staging       # requires environment protection rules

    steps:
      - uses: actions/checkout@v4
      - name: Deploy CDKTF to staging
        run: cdktf deploy va-pipeline-staging --auto-approve
        env:
          GOOGLE_CREDENTIALS: ${{ secrets.GCP_SA_KEY_STAGING }}
          GCP_PROJECT_ID: ${{ secrets.GCP_PROJECT_ID }}

  deploy-prod:
    name: Deploy to Production
    runs-on: ubuntu-latest
    if: github.event_name == 'workflow_dispatch' && github.event.inputs.environment == 'prod'
    environment: prod          # requires manual approval in GitHub

    steps:
      - uses: actions/checkout@v4
      - name: Deploy CDKTF to prod
        run: cdktf deploy va-pipeline-prod --auto-approve
        env:
          GOOGLE_CREDENTIALS: ${{ secrets.GCP_SA_KEY_PROD }}
          GCP_PROJECT_ID: ${{ secrets.GCP_PROJECT_ID }}
```

**Environment protection rules** in GitHub (Settings → Environments → prod) let you require a human approver before the prod job runs. The CD pipeline pauses, sends a Slack/email notification, and waits.

In [ ]:
# ============================================================
# SECTION 7 -- Environment Promotion: Simulating config and promotion
# Environment promotion = the controlled progression dev -> staging -> prod
# CD = Continuous Deployment (automated pipeline that ships approved code)
# ============================================================
# WHY have multiple environments?
# dev     = developer sandbox. Break things freely. Auto-deploy on every push.
# staging = pre-production mirror. Full dataset size. Catch integration bugs.
# prod    = live system. Real data. Manual approval gate before any deploy.
# Keeping them separate prevents dev experiments from corrupting prod data.

import json
from datetime import datetime

# ─────────────────────────────────────────────────────────────────────────────
# Environment configuration map -- single source of truth for all env settings
# WHY store this as a dict instead of separate variables?
# It makes env-specific logic a dict lookup (ENV_CONFIG[env]) instead of
# a chain of if/elif branches -- easier to read and extend.
# ─────────────────────────────────────────────────────────────────────────────
ENV_CONFIG = {
    "dev": {
        "gcs_bronze_bucket":   "quintrix-va-bronze-dev",
        # BigQuery dataset -- each env has its own to prevent cross-contamination
        "bq_gold_dataset":     "gold_dev",
        # Dataproc = GCP managed Spark cluster service
        "dataproc_cluster":    "va-pipeline-dev",
        # Preemptible (spot) workers are cheaper but can be evicted mid-job.
        # Dev jobs are short enough that eviction risk is acceptable.
        "preemptible_workers": 0,
        "sample_size":         1000,    # small sample keeps dev runs fast
        "auto_deploy":         True,    # no human needed for dev deploys
        "requires_approval":   False,
    },
    "staging": {
        "gcs_bronze_bucket":   "quintrix-va-bronze-staging",
        "bq_gold_dataset":     "gold_staging",
        "dataproc_cluster":    "va-pipeline-staging",
        "preemptible_workers": 2,       # moderate spot workers for cost savings
        "sample_size":         50000,   # large enough to catch performance issues
        "auto_deploy":         True,
        "requires_approval":   False,
    },
    "prod": {
        "gcs_bronze_bucket":   "quintrix-va-bronze-prod",
        "bq_gold_dataset":     "gold_prod",
        "dataproc_cluster":    "va-pipeline-prod",
        "preemptible_workers": 4,       # spot workers save ~60-80% on large jobs
        "sample_size":         None,    # None = process the full dataset
        "auto_deploy":         False,   # prod NEVER auto-deploys
        "requires_approval":   True,    # Tech Lead must click Approve in GitHub
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# Promotion simulation function
# WHY simulate? Real promotions involve GitHub webhook calls, Slack
# notifications, and cloud CLI commands. This function models the decision
# logic so you understand what each CI/CD step does.
# ─────────────────────────────────────────────────────────────────────────────
def simulate_promotion(change_description: str, ci_passes: bool):
    """
    Walk through a full dev -> staging -> prod promotion.
    ci_passes: set False to see the PR-blocked path.
    """
    print("=" * 60)
    print(f"  PROMOTION SIMULATION")
    print(f"  Change: {change_description}")
    print("=" * 60)

    # CI (Continuous Integration) steps run automatically on every PR push.
    # These checks must all pass before a human reviewer even sees the PR.
    steps = [
        ("Developer pushes branch",        True),
        ("PR opened",                       True),
        # linting = automated code style checks (ruff, flake8)
        ("CI: lint + unit tests",           ci_passes),
        # DAG = Directed Acyclic Graph (Airflow pipeline definition file)
        ("CI: DAG validation",              ci_passes),
        # CDKTF plan shows what infrastructure changes the PR would make
        ("CI: CDKTF plan (staging diff)",   ci_passes),
        # dry-run = run quality gate checks without writing any data
        ("CI: data quality dry-run",        ci_passes),
    ]

    ci_ok = True
    for step, passes in steps:
        icon = "+" if passes else "x"    # + = pass, x = fail
        print(f"  [{icon}] {step}")
        if not passes:
            ci_ok = False
            break   # stop on first failure -- no point running later steps

    if not ci_ok:
        print()
        print("  CI FAILED -- PR blocked. Reviewer never sees it.")
        print("  Developer fixes the issue, pushes again, CI re-runs.")
        return

    # CD (Continuous Deployment) steps run only after CI and code review pass
    print()
    print("  All CI checks passed. PR approved by reviewer.")
    print()

    for env in ["staging", "prod"]:
        cfg = ENV_CONFIG[env]

        # WHY require manual approval for prod?
        # Automated systems cannot judge business impact. A human must verify
        # that the change is safe to ship to live customer data.
        if cfg["requires_approval"] and env == "prod":
            print(f"  [WAITING] prod deploy -- awaiting manual approval from Tech Lead")
            print(f"            GitHub sends Slack notification to #deployments")
            approved = True   # simulate: Tech Lead clicks Approve
            if not approved:
                print("  [REJECTED] Deploy cancelled.")
                return
            print("  [APPROVED] Tech Lead approved. Proceeding with prod deploy.")
        else:
            print(f"  [AUTO] Deploying to {env}...")

        # Show the actual CLI commands that CD would run
        print(f"    Bucket : {cfg['gcs_bronze_bucket']}")
        print(f"    Cluster: {cfg['dataproc_cluster']} ({cfg['preemptible_workers']} preemptible workers)")
        print(f"    Dataset: {cfg['bq_gold_dataset']}")
        # --auto-approve skips the interactive confirmation -- required in CI
        print(f"    cdktf deploy va-pipeline-{env} --auto-approve")
        print()

    print("  Promotion complete: dev -> staging -> prod")

# ─────────────────────────────────────────────────────────────────────────────
# Run both scenarios -- happy path and CI failure path
# ─────────────────────────────────────────────────────────────────────────────

# You should see: all CI steps pass, staging auto-deploys, prod waits for approval
simulate_promotion("Fix UTC timezone bug in bronze_ingest.py", ci_passes=True)
print()
print("-" * 60)
print()
# You should see: CI fails at lint+unit tests, PR is blocked, deploy never starts
simulate_promotion("Rename outcome to call_result in silver_transform.py", ci_passes=False)

---
## 8. Data Quality Gates in CI <a id="8-quality-gates"></a>

> **In plain English:** The factory inspector who stops the assembly line when a part is defective. In a car factory, if a door doesn't fit, the inspector stops the line — not the manager, not the customer. Bad data should never reach the Gold mart. A quality gate in CI is that inspector: it runs before merge and blocks the PR if data quality rules fail.

---

### What to Validate in CI

| Check | What It Catches | Example |
|---|---|---|
| **Schema validation** | Column renames, type changes | `converted` column missing after a refactor |
| **Row count sanity** | Lost 90% of rows due to a bad join | 50,000 calls → 500 after Silver transform |
| **Null percentage** | Nulls exploded due to source change | `call_date` is 40% null (was always 0%) |
| **Referential integrity** | Broken foreign keys | `orders.call_id` references non-existent `calls.call_id` |
| **Value range** | Out-of-range values | `duration_sec = -30` |
| **Duplicate detection** | ETL (Extract, Transform, Load) ran twice | 100,000 rows instead of 50,000 |

---

### Quality Gate Implementation

```python
# scripts/run_quality_checks.py
import sys
import json
import argparse
import pandas as pd
from datetime import date

def load_sample_data():
    """In CI: load sample data from fixtures. In prod: load from GCS/S3."""
    # Simulate Silver output
    return pd.DataFrame([
        {"call_id": "C001", "call_date": "2026-03-20", "duration_min": 2.0,
         "converted": 1, "campaign_key": 1},
        {"call_id": "C002", "call_date": "2026-03-20", "duration_min": 5.5,
         "converted": 0, "campaign_key": 2},
        {"call_id": "C003", "call_date": None,           "duration_min": 1.2,
         "converted": 0, "campaign_key": 1},
    ])

class QualityGate:
    def __init__(self, df: pd.DataFrame, dry_run: bool = False):
        self.df = df
        self.dry_run = dry_run
        self.results = []

    def check_schema(self, expected_columns: set) -> bool:
        actual = set(self.df.columns)
        missing = expected_columns - actual
        extra   = actual - expected_columns
        passed = len(missing) == 0
        self.results.append({
            "check": "schema",
            "passed": passed,
            "detail": f"Missing: {missing}, Extra: {extra}" if not passed else "OK",
        })
        return passed

    def check_row_count(self, min_rows: int, max_drop_pct: float = 0.5) -> bool:
        n = len(self.df)
        passed = n >= min_rows
        self.results.append({
            "check": "row_count",
            "passed": passed,
            "detail": f"{n} rows (min: {min_rows})" if not passed else f"{n} rows OK",
        })
        return passed

    def check_null_pct(self, column: str, max_null_pct: float) -> bool:
        null_pct = self.df[column].isna().mean()
        passed = null_pct <= max_null_pct
        self.results.append({
            "check": f"null_pct:{column}",
            "passed": passed,
            "detail": f"{null_pct:.1%} nulls (max allowed: {max_null_pct:.1%})",
        })
        return passed

    def check_value_range(self, column: str, min_val, max_val) -> bool:
        out_of_range = self.df[
            (self.df[column] < min_val) | (self.df[column] > max_val)
        ]
        passed = len(out_of_range) == 0
        self.results.append({
            "check": f"range:{column}",
            "passed": passed,
            "detail": f"{len(out_of_range)} rows out of range [{min_val}, {max_val}]"
                      if not passed else f"All values in [{min_val}, {max_val}]",
        })
        return passed

    def check_no_duplicates(self, key_columns: list) -> bool:
        dupe_count = self.df.duplicated(subset=key_columns).sum()
        passed = dupe_count == 0
        self.results.append({
            "check": "duplicates",
            "passed": passed,
            "detail": f"{dupe_count} duplicate rows on {key_columns}",
        })
        return passed

    def run_all(self) -> bool:
        EXPECTED_COLUMNS = {"call_id", "call_date", "duration_min",
                             "converted", "campaign_key"}
        checks = [
            self.check_schema(EXPECTED_COLUMNS),
            self.check_row_count(min_rows=1),
            self.check_null_pct("call_id",   max_null_pct=0.0),
            self.check_null_pct("call_date",  max_null_pct=0.05),
            self.check_value_range("duration_min", 0, 120),
            self.check_value_range("converted",    0, 1),
            self.check_no_duplicates(["call_id"]),
        ]
        return all(checks)

    def report(self) -> str:
        lines = ["Data Quality Gate Report", "=" * 40]
        for r in self.results:
            icon = "PASS" if r["passed"] else "FAIL"
            lines.append(f"  [{icon}] {r['check']}: {r['detail']}")
        all_passed = all(r["passed"] for r in self.results)
        lines.append("=" * 40)
        lines.append(f"Overall: {'PASS — gate open' if all_passed else 'FAIL — gate CLOSED'}")
        return "\n".join(lines)
```

---

### Adding the Quality Gate to GitHub Actions

```yaml
# In .github/workflows/ci.yml — add this step to data-quality job:

      - name: Run quality checks on sample data
        run: |
          python scripts/run_quality_checks.py --dry-run
          # Exit code 1 if any check fails → CI step fails → PR blocked
```

```python
# At the bottom of run_quality_checks.py:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dry-run", action="store_true")
    args = parser.parse_args()

    df = load_sample_data()
    gate = QualityGate(df, dry_run=args.dry_run)
    all_passed = gate.run_all()
    print(gate.report())
    sys.exit(0 if all_passed else 1)   # ← exit code 1 fails the CI step
```

---

### Break the System Exercise

**Scenario:** Your quality gate blocks a deploy at 5:00 PM on Friday. The VP wants updated campaign numbers by Monday morning. The failing check: `null_pct:call_date` is at 8% (max allowed: 5%).

**Discussion questions:**
1. Is the right response to raise the threshold to 10%? Why or why not?
2. What is the root cause you should investigate before any deploy decision?
3. If the source system had a known 4-hour outage today and the nulls are expected to self-heal, what is the right process to get to prod safely?

**Architect's Note:** Quality gates are only useful if they have teeth. A gate that gets bypassed under pressure teaches the team that quality is optional. The right answer is: investigate root cause, fix or document the exception with a time-bound ticket, then deploy with explicit sign-off. Never silently raise the threshold.

In [ ]:
# ============================================================
# SECTION 8 -- Data Quality Gates: Live demo
# Quality gate = automated checks that must pass before data advances
#                to the next medallion layer (Bronze -> Silver -> Gold)
# CI  = Continuous Integration -- quality gates run inside the CI pipeline
# ============================================================
# WHY quality gates in CI/CD and not just in the pipeline?
# Running them in CI means a bad schema change or a source-system outage
# is caught on the PR -- before the change ships -- not at 3 AM when the
# dashboard is wrong and the VP is asking questions.

import pandas as pd
import numpy as np    # used later for NaN injection

# ─────────────────────────────────────────────────────────────────────────────
# HELLO-WORLD RAMP: the simplest possible quality check
# A quality check is just a function that reads a DataFrame and returns
# True (pass) or False (fail) with a human-readable reason.
# ─────────────────────────────────────────────────────────────────────────────
def simple_null_check(df, col):
    """Returns True if column has zero nulls, False otherwise."""
    null_count = df[col].isna().sum()
    # You should see: null_count=0 for the sample data below
    print(f"  simple_null_check({col}): {null_count} nulls -> {'PASS' if null_count == 0 else 'FAIL'}")
    return null_count == 0

sample = pd.DataFrame([{"call_id": "C001", "call_date": "2026-03-20"}])
simple_null_check(sample, "call_id")
print()

# ─────────────────────────────────────────────────────────────────────────────
# Full QualityGate class -- wraps multiple checks into a single report
# WHY a class instead of standalone functions?
# The class accumulates results across all checks so you can print a
# single pass/fail report at the end, matching what Great Expectations
# and dbt tests produce in production pipelines.
# ─────────────────────────────────────────────────────────────────────────────
class QualityGate:
    def __init__(self, df: pd.DataFrame, label: str = ""):
        self.df      = df
        self.label   = label
        self.results = []   # list of {check, passed, detail} dicts

    def check(self, name: str, passed: bool, detail: str):
        """Record one check result. Returns the passed flag for chaining."""
        self.results.append({"check": name, "passed": passed, "detail": detail})
        return passed

    def check_schema(self, expected_columns: set) -> bool:
        """Fail if any expected column is missing or an unexpected one appears."""
        actual  = set(self.df.columns)
        missing = expected_columns - actual   # columns the downstream job needs
        extra   = actual - expected_columns   # columns that should not be there
        ok = len(missing) == 0
        return self.check("schema", ok,
            "OK" if ok else f"Missing={missing} Extra={extra}")

    def check_null_pct(self, col: str, max_pct: float) -> bool:
        """Fail if the null percentage for a column exceeds max_pct.

        BEGINNER NOTE: .mean() on a boolean Series returns the fraction of True
        values. isna() returns True for null/NaN, so .isna().mean() = null rate.
        """
        if col not in self.df.columns:
            return self.check(f"null:{col}", False, f"Column '{col}' not found")
        pct = self.df[col].isna().mean()
        ok  = pct <= max_pct
        return self.check(f"null:{col}", ok,
            f"{pct:.1%} (max {max_pct:.1%})" if not ok else f"{pct:.1%} OK")

    def check_row_count(self, minimum: int) -> bool:
        """Fail if the DataFrame has fewer rows than the minimum threshold."""
        n  = len(self.df)
        ok = n >= minimum
        return self.check("row_count", ok,
            f"{n} rows OK" if ok else f"{n} rows < minimum {minimum}")

    def check_duplicates(self, keys: list) -> bool:
        """Fail if any combination of key columns is duplicated.

        WHY check for duplicates?
        Duplicate call_ids in Silver cause double-counting in Gold aggregations
        -- conversion rates appear inflated, campaigns look more effective than
        they actually are.
        """
        dupes = self.df.duplicated(subset=keys).sum()
        ok    = dupes == 0
        return self.check("duplicates", ok,
            "No duplicates" if ok else f"{dupes} duplicate rows on {keys}")

    def check_range(self, col: str, lo, hi) -> bool:
        """Fail if any value in col is outside the [lo, hi] range.

        BEGINNER NOTE: The | operator between two boolean Series is
        element-wise OR -- it flags rows where EITHER condition is true.
        """
        if col not in self.df.columns:
            return self.check(f"range:{col}", False, f"Column '{col}' not found")
        # Count rows where value is below lo OR above hi
        bad = ((self.df[col] < lo) | (self.df[col] > hi)).sum()
        ok  = bad == 0
        return self.check(f"range:{col}", ok,
            f"All in [{lo},{hi}]" if ok else f"{bad} rows outside [{lo},{hi}]")

    def run(self, expected_cols) -> bool:
        """Run all checks in sequence. Returns True only if every check passes."""
        results = [
            self.check_schema(expected_cols),
            self.check_row_count(1),
            # call_id must never be null -- it is the primary key
            self.check_null_pct("call_id",   0.00),
            # call_date: tolerate up to 5% nulls (source outages happen)
            self.check_null_pct("call_date", 0.05),
            # duration_min: 0-120 minutes is the valid range for a call
            self.check_range("duration_min", 0, 120),
            # converted must be 0 or 1 -- any other value is a bug
            self.check_range("converted",    0, 1),
            self.check_duplicates(["call_id"]),
        ]
        return all(results)

    def report(self) -> None:
        """Print a formatted pass/fail report for all accumulated checks."""
        print(f"\n{'=' * 55}")
        print(f"  Quality Gate Report: {self.label}")
        print(f"{'=' * 55}")
        for r in self.results:
            icon = "PASS" if r["passed"] else "FAIL"
            print(f"  [{icon}] {r['check']}: {r['detail']}")
        passed = all(r["passed"] for r in self.results)
        print(f"{'=' * 55}")
        print(f"  Overall: {'PASS -- deploy unblocked' if passed else 'FAIL -- deploy BLOCKED'}")
        return passed

# The expected Silver schema -- shared by all three scenarios below
EXPECTED_COLS = {"call_id", "call_date", "duration_min", "converted", "campaign_key"}

# ─────────────────────────────────────────────────────────────────────────────
# Scenario 1: Clean data -- all checks should pass
# ─────────────────────────────────────────────────────────────────────────────
df_clean = pd.DataFrame([
    {"call_id": "C001", "call_date": "2026-03-20", "duration_min": 2.0,  "converted": 1, "campaign_key": 1},
    {"call_id": "C002", "call_date": "2026-03-20", "duration_min": 5.5,  "converted": 0, "campaign_key": 2},
    {"call_id": "C003", "call_date": "2026-03-20", "duration_min": 10.3, "converted": 1, "campaign_key": 3},
])
gate1 = QualityGate(df_clean, "Clean Silver data")
gate1.run(EXPECTED_COLS)
# You should see: all PASS, Overall: PASS -- deploy unblocked
gate1.report()

# ─────────────────────────────────────────────────────────────────────────────
# Scenario 2: Source outage caused ~8% null call_date (exceeds 5% threshold)
# WHY is 8% > threshold realistic?
# If the LT (ListenTrust) API goes down for 4 hours in a 50-hour week,
# roughly 8% of call records arrive with no date stamp.
# ─────────────────────────────────────────────────────────────────────────────
df_with_nulls = pd.DataFrame([
    {"call_id": f"C{i:03d}", "call_date": None if i % 12 == 0 else "2026-03-20",
     "duration_min": 3.0, "converted": 0, "campaign_key": 1}
    for i in range(1, 26)
])
gate2 = QualityGate(df_with_nulls, "Data after 4-hr source outage (8% null call_date)")
gate2.run(EXPECTED_COLS)
# You should see: null:call_date FAIL because 8.3% > 5% threshold
gate2.report()

# ─────────────────────────────────────────────────────────────────────────────
# Scenario 3: Column rename bug -- converted replaced with call_result
# WHY test this explicitly? It is the most common schema drift bug in
# real pipelines and the hardest to catch without automated schema checks.
# ─────────────────────────────────────────────────────────────────────────────
df_renamed = pd.DataFrame([
    {"call_id": "C001", "call_date": "2026-03-20", "duration_min": 2.0,
     "call_result": "converted", "campaign_key": 1},  # converted was renamed
])
gate3 = QualityGate(df_renamed, "Silver after column rename bug")
gate3.run(EXPECTED_COLS)
# You should see: schema FAIL -- Missing={'converted'} Extra={'call_result'}
gate3.report()

---
## 9. DAG Deployment — GitOps for Airflow <a id="9-dag-deployment"></a>

> **In plain English:** Instead of SSH-ing into the Airflow server and uploading DAG files by hand, your DAGs deploy automatically when you merge a PR. The same way app code deploys to a web server on merge, your DAG deploys to Cloud Composer or MWAA on merge. Nobody needs server access. Nobody forgets to upload. Nobody uploads the wrong version.

**This pattern is called GitOps:** Git is the single source of truth. What's in Git is what runs in production.

---

### The Problem GitOps Solves

**Before GitOps:**
```
Developer edits DAG locally
  ↓
Runs: gcloud composer environments storage dags import ...
       (or: aws s3 cp dags/va_analytics_pipeline.py s3://mwaa-bucket/dags/)
  ↓
Easy to forget. Easy to upload the wrong file.
No history of who deployed what and when.
No review. No tests.
```

**With GitOps:**
```
Developer edits DAG → opens PR → CI validates it → reviewer approves → merges
  ↓
GitHub Actions automatically syncs dags/ folder to Cloud Composer / MWAA
  ↓
Airflow picks up the new DAG within 30-60 seconds (automatic folder scan)
  ↓
Full audit trail in Git: who changed what, when, why (PR description)
```

---

### Full GitOps Workflow for DAGs

```
1. Developer: git checkout -b fix/dag-add-data-quality-task
2. Developer: edits dags/va_analytics_pipeline.py
3. Developer: git push → opens PR

4. CI runs (triggered by PR):
   ├── Lint: ruff check dags/va_analytics_pipeline.py
   ├── Import check: python -c "import va_analytics_pipeline"
   ├── Parse check: airflow dags list (no import errors?)
   └── Dry-run: airflow tasks test va_analytics_pipeline bronze_ingest 2026-03-01

5. Reviewer approves → merge to main

6. CD runs (triggered by merge to main):
   GCP path: gsutil -m rsync -r dags/ gs://composer-bucket/dags/
   AWS path: aws s3 sync dags/ s3://mwaa-bucket/dags/

7. Airflow's DAG folder scanner picks up changes (every 30s by default)
8. New DAG version active. No server access required.
```

---

### GitHub Actions: DAG CI + CD

```yaml
# .github/workflows/dag-cicd.yml
name: DAG CI/CD

on:
  pull_request:
    paths:
      - 'dags/**'            # only run when DAG files change
  push:
    branches: [ main ]
    paths:
      - 'dags/**'

jobs:
  # ─── CI: Validate DAGs ─────────────────────────────────────────
  dag-ci:
    name: Validate DAGs
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install Airflow + providers
        run: |
          pip install apache-airflow==2.9.0             apache-airflow-providers-google             apache-airflow-providers-amazon             --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.0/constraints-3.11.txt"

      - name: Lint DAG files
        run: ruff check dags/

      - name: Check DAG imports
        run: |
          for dag_file in dags/*.py; do
            echo "Checking: $dag_file"
            python -c "
          import importlib.util, sys
          spec = importlib.util.spec_from_file_location('dag', '$dag_file')
          mod  = importlib.util.module_from_spec(spec)
          spec.loader.exec_module(mod)
          print('  Import OK:', '$dag_file')
          "
          done

      - name: Parse DAGs with Airflow
        run: |
          export AIRFLOW__CORE__DAGS_FOLDER=$PWD/dags
          export AIRFLOW__DATABASE__SQL_ALCHEMY_CONN=sqlite:///airflow_ci.db
          airflow db init
          airflow dags list
          # Fail if any import errors
          ERRORS=$(airflow dags list-import-errors --output json 2>/dev/null || echo '[]')
          COUNT=$(echo $ERRORS | python3 -c "import json,sys; print(len(json.load(sys.stdin)))")
          if [ "$COUNT" -gt 0 ]; then
            echo "DAG import errors found:"
            echo $ERRORS
            exit 1
          fi
          echo "No DAG import errors."

  # ─── CD: Deploy DAGs to Cloud Composer ─────────────────────────
  dag-deploy-gcp:
    name: Deploy DAGs to Cloud Composer
    runs-on: ubuntu-latest
    needs: dag-ci
    if: github.ref == 'refs/heads/main' && github.event_name == 'push'
    environment: staging

    steps:
      - uses: actions/checkout@v4

      - name: Authenticate to GCP
        uses: google-github-actions/auth@v2
        with:
          credentials_json: ${{ secrets.GCP_SA_KEY_STAGING }}

      - name: Set up gcloud CLI
        uses: google-github-actions/setup-gcloud@v2

      - name: Sync DAGs to Cloud Composer
        run: |
          COMPOSER_BUCKET=$(gcloud composer environments describe va-composer-staging             --location us-central1             --format "value(config.dagGcsPrefix)")
          echo "Syncing to: $COMPOSER_BUCKET"
          gsutil -m rsync -r -d dags/ $COMPOSER_BUCKET/
          echo "Sync complete. Airflow will pick up changes within 60 seconds."

  # ─── CD: Deploy DAGs to MWAA (AWS) ─────────────────────────────
  dag-deploy-aws:
    name: Deploy DAGs to MWAA
    runs-on: ubuntu-latest
    needs: dag-ci
    if: github.ref == 'refs/heads/main' && github.event_name == 'push'
    environment: staging

    steps:
      - uses: actions/checkout@v4

      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v4
        with:
          aws-access-key-id:     ${{ secrets.AWS_ACCESS_KEY_ID }}
          aws-secret-access-key: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          aws-region: us-east-1

      - name: Sync DAGs to MWAA S3 bucket
        run: |
          aws s3 sync dags/ s3://${{ secrets.MWAA_DAG_BUCKET }}/dags/             --delete             --exclude "*.pyc"             --exclude "__pycache__/*"
          echo "DAGs synced. MWAA will pick up changes within 30 seconds."
```

---

### DAG Versioning and Rollback

**The GitOps way to roll back:**

```bash
# If the deployed DAG breaks production:

# 1. Find the last good commit
git log --oneline dags/va_analytics_pipeline.py

# 2. Create a revert PR (don't force push)
git revert HEAD
git push origin revert/bad-dag-change
# Open PR → CI validates the reverted DAG → merge → CD deploys the old version

# Never: git push --force main
# Always: revert commit → PR → review → merge → deploy
```

---

### Canary Deploys for DAGs

For high-stakes DAG changes, deploy to one environment first and validate before promoting:

```yaml
# Canary pattern for DAGs
1. Deploy to dev Cloud Composer → run va_analytics_pipeline for 1 day
2. Validate: Gold mart row counts match expected range
3. Deploy to staging → run for 3 days
4. Validate: compare Gold mart output to previous week (same date range)
5. Deploy to prod
```

---

### Exercise: Explain Like I'm CEO

**Close the notebook. Answer out loud (90 seconds):**

The CTO asks: *"We're talking about GitOps. What does that actually mean for our pipeline team?"*

> "It means Git is the only way anything reaches production. No one SSH-es into servers. No one uploads files by hand. Every change — a DAG update, a transform fix, an infrastructure change — goes through a pull request. The PR gets reviewed, tested automatically, and then deployed by a robot. If something breaks in production, you find the PR that caused it, revert it in 60 seconds, and production is back. You get a complete audit trail: who changed what, when, and why. It also means any engineer on the team can deploy safely — they don't need to know the gcloud commands or the S3 bucket names. They just merge a PR."

---

### Exercise: Break the System

| Scenario | What Breaks | How to Recover |
|---|---|---|
| Bad DAG merged — Airflow shows import error | Scheduler stops picking up tasks for that DAG | Revert the commit → new PR → CI validates → merge → re-deploy |
| CD job uploads to wrong Composer bucket | Prod runs old DAG; staging has new one | Fix the `COMPOSER_BUCKET` lookup; re-run CD job manually |
| Two engineers merge to main within 30 seconds | gsutil rsync race condition — one DAG version wins | Remote state locking (GCS bucket versioning) or serialized deploys |
| DAG change removes a task that had running instances | TaskInstances in RUNNING state have no task definition | Always keep old task_id for 1 cycle when removing tasks; use `trigger_rule` |

---

### Edge Cases

- **DAG versioning:** Airflow 2.x supports DAG versioning natively — old runs reference the DAG definition at their start time
- **`--delete` flag in gsutil rsync:** Removes files from Composer that no longer exist in your repo — use carefully; a bug in CI could wipe all DAGs
- **parse check in CI:** `airflow dags list-import-errors` exits 0 even with errors — always parse the JSON output and check the count explicitly

In [ ]:
# ============================================================
# SECTION 9 -- DAG Deployment: Simulating GitOps workflow
# DAG     = Directed Acyclic Graph (an Airflow pipeline definition file)
# GitOps  = using Git as the single source of truth for both code AND
#           infrastructure/config -- all changes go through a PR
# CD      = Continuous Deployment (automated deploy on merge to main)
# Airflow = the workflow orchestration tool that reads and executes DAGs
# ============================================================
# WHY GitOps for DAG deployment?
# Without GitOps, engineers SSH into the Airflow server and manually copy
# DAG files -- no audit trail, easy to overwrite each other's work.
# With GitOps: every DAG change is a PR, CI validates the file, CD deploys
# it automatically on merge. Full audit trail lives in Git history.

import ast          # ast = Abstract Syntax Tree (Python built-in parser module)
import textwrap
from datetime import datetime

# ─────────────────────────────────────────────────────────────────────────────
# HELLO-WORLD RAMP: the minimal valid Airflow DAG
# WHY show a short example before the full DAG?
# The real VA Analytics DAG has 4 tasks and multiple imports -- intimidating
# if you have never written one. This minimal version proves the basic structure.
# ─────────────────────────────────────────────────────────────────────────────
HELLO_WORLD_DAG = '''
from airflow import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime

# A DAG file must create a DAG object -- Airflow scans for it automatically
with DAG(
    dag_id="hello_world",        # unique identifier shown in the Airflow UI
    start_date=datetime(2026, 3, 1),
    schedule="@daily",           # run once per day
    catchup=False,               # do not backfill missed runs
) as dag:
    say_hello = BashOperator(
        task_id="say_hello",
        bash_command="echo Hello from Airflow!",
    )
'''
print("Hello-world DAG:")
print(HELLO_WORLD_DAG)

# ─────────────────────────────────────────────────────────────────────────────
# Full production DAG -- the VA Analytics pipeline
# Four tasks in sequence: Bronze ingest -> Silver transform -> quality check -> Gold
# ─────────────────────────────────────────────────────────────────────────────
GOOD_DAG = '''
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator
from datetime import datetime

def run_quality_check(**ctx):
    print(f"Running quality check for {ctx['ds']}")
    return "quality_ok"

with DAG(
    dag_id="va_analytics_pipeline",
    start_date=datetime(2026, 3, 1),
    schedule="@daily",
    catchup=False,
    tags=["call-center", "va", "production"],
) as dag:

    bronze = BashOperator(
        task_id="bronze_ingest",
        bash_command="python /opt/airflow/scripts/bronze_ingest.py --date {{ ds }}",
    )

    silver = BashOperator(
        task_id="silver_transform",
        bash_command="python /opt/airflow/scripts/silver_transform.py --date {{ ds }}",
    )

    quality = PythonOperator(
        task_id="data_quality_check",
        python_callable=run_quality_check,
    )

    gold = BashOperator(
        task_id="gold_aggregate",
        bash_command="python /opt/airflow/scripts/gold_aggregate.py --date {{ ds }}",
    )

    bronze >> silver >> quality >> gold
'''

# ─────────────────────────────────────────────────────────────────────────────
# BAD DAG examples -- CI should catch these before they reach Airflow
# ─────────────────────────────────────────────────────────────────────────────

# Bad DAG 1: syntax error (missing comma)
BAD_DAG_SYNTAX = '''
from airflow import DAG
from datetime import datetime

with DAG(
    dag_id="va_analytics_pipeline"
    # Missing comma above -- Python will raise SyntaxError
    start_date=datetime(2026, 3, 1),
) as dag:
    pass
'''

# Bad DAG 2: imports a non-existent operator
# WHY is this a problem if it is valid Python syntax?
# Airflow loads every DAG file on startup. An ImportError in one file
# can prevent ALL DAGs from loading -- the entire scheduler goes down.
BAD_DAG_IMPORT = '''
from airflow import DAG
from airflow.operators.nonexistent import FakeOperator  # does not exist
from datetime import datetime

with DAG("va_analytics_pipeline", start_date=datetime(2026,3,1)) as dag:
    t = FakeOperator(task_id="fake")
'''

# ─────────────────────────────────────────────────────────────────────────────
# CI validation functions -- these run in the GitHub Actions workflow
# ─────────────────────────────────────────────────────────────────────────────

def check_dag_syntax(dag_source: str, label: str) -> bool:
    """Simulate what CI does: parse the DAG file for syntax errors.

    ast.parse() = Python built-in parser. It converts source code into
    an Abstract Syntax Tree (AST) without executing the code. Any SyntaxError
    is caught here -- safe because no imports or network calls happen.
    """
    print(f"  Checking syntax: {label}")
    try:
        tree = ast.parse(dag_source)
        # Count top-level statements as a basic complexity metric
        print(f"    [PASS] Syntax OK ({len(tree.body)} top-level statements)")
        return True
    except SyntaxError as e:
        # e.lineno and e.msg come directly from the Python parser
        print(f"    [FAIL] SyntaxError at line {e.lineno}: {e.msg}")
        return False

def check_dag_has_dag_object(dag_source: str, label: str) -> bool:
    """Simulate checking that a DAG object is defined in the file.

    Airflow's DAG loader looks for a variable of type DAG in each file.
    If none exists, the file is skipped silently -- a common beginner mistake.
    """
    has_dag_id = 'dag_id=' in dag_source or 'DAG(' in dag_source
    print(f"  Checking for DAG object: {label}")
    if has_dag_id:
        print(f"    [PASS] DAG definition found")
    else:
        print(f"    [FAIL] No DAG object found in file")
    return has_dag_id

# ─────────────────────────────────────────────────────────────────────────────
# GitOps workflow simulation -- PR -> CI -> merge -> CD -> Cloud Composer
# Cloud Composer = GCP managed Airflow service (fully hosted)
# ─────────────────────────────────────────────────────────────────────────────
def simulate_gitops_workflow(dag_source: str, label: str, branch: str):
    print("=" * 60)
    print(f"  GitOps Workflow: {label}")
    print(f"  Branch: {branch}")
    print("=" * 60)

    # CI checks run automatically when the PR is opened
    print("\n[CI] Pull Request opened -- running DAG validation...")
    syntax_ok  = check_dag_syntax(dag_source, label)
    # WHY skip check_dag_has_dag_object if syntax fails?
    # Parsing the AST on a file with SyntaxError would raise an exception.
    # Short-circuit: no point running later checks on broken syntax.
    dag_obj_ok = check_dag_has_dag_object(dag_source, label) if syntax_ok else False

    ci_passed = syntax_ok and dag_obj_ok

    if not ci_passed:
        print("\n[CI FAIL] PR blocked. DAG validation failed.")
        print("         Developer must fix the DAG and push again.")
        return

    # All CI checks passed -- human reviewer approves the PR
    print("\n[CI PASS] All checks passed.")
    print("[REVIEW]  Reviewer approves PR.")
    print("[MERGE]   PR merged to main.")
    print()

    # CD (Continuous Deployment): gsutil syncs dags/ folder to Cloud Composer
    print("[CD] Deploying DAG to Cloud Composer...")
    # gsutil rsync = Google Cloud Storage sync command (like rsync but for GCS)
    # -r = recursive, -d = delete files in destination not in source
    print(f"     gsutil -m rsync -r -d dags/ gs://composer-va-staging/dags/")
    print(f"     Uploaded: {label}.py")
    # Cloud Composer (Airflow) re-scans the dags/ folder on an interval
    print(f"     Airflow DAG folder scanner interval: 30s")
    print(f"     Status: DAG '{label}' will be active within 30 seconds")
    print()
    print(f"[DONE] {label} is live in Cloud Composer staging.")

# ─────────────────────────────────────────────────────────────────────────────
# Run the GitOps simulation for good and bad DAGs
# ─────────────────────────────────────────────────────────────────────────────

# You should see: CI passes, CD deploys successfully
simulate_gitops_workflow(GOOD_DAG, "va_analytics_pipeline", "feature/add-quality-check")

print()
# You should see: CI fails on syntax error, PR is blocked, no deploy happens
simulate_gitops_workflow(BAD_DAG_SYNTAX, "va_analytics_pipeline", "fix/bad-dag")

# ─────────────────────────────────────────────────────────────────────────────
# Extract and display the task dependency chain from the good DAG
# WHY show this? In Airflow's Graph view you see this as a visual flow chart.
# Understanding the chain helps you diagnose failures: if Gold fails,
# check quality first, then Silver, then Bronze.
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("  Task dependency chain extracted from good DAG:")
print("=" * 60)
tasks = ["bronze_ingest", "silver_transform", "data_quality_check", "gold_aggregate"]
for i, task in enumerate(tasks):
    # Print with arrow between tasks, no arrow after the last task
    arrow = " -> " if i < len(tasks) - 1 else ""
    print(f"  {task}{arrow}", end="")
print()
print()
print("This is what Airflow renders in the Graph view.")
# You should see: bronze_ingest -> silver_transform -> data_quality_check -> gold_aggregate
print("CI validates this chain exists and imports correctly before any deploy.")

---
## Key Takeaways + Homework <a id="takeaways"></a>

### Key Takeaways (9 Sections → 12 Rules)

1. **CI/CD is not optional for data engineers** — your pipeline code, SQL transforms, DAGs, and infrastructure definitions are all application code. They all need automated testing.

2. **IaC manages state; CLI scripts don't** — Terraform and CDKTF know what exists, what changed, and what to destroy. A shell script doesn't. Use IaC.

3. **Read Terraform HCL** — you will encounter it at every data engineering company. You don't need to master it. You need to read `resource`, `variable`, `output`, and understand `plan` vs `apply`.

4. **CDKTF = Python + Terraform ecosystem** — write Python, get multi-cloud IaC with the full Terraform provider ecosystem. Best choice for Python-first teams on GCP or multi-cloud.

5. **AWS CDK = Python + CloudFormation** — same DX as CDKTF, AWS-only. CloudFormation handles state internally.

6. **CI blocks bad code before merge** — linting, unit tests, DAG validation, schema checks, and CDKTF plan all run before a reviewer even sees the PR. The robot is the first reviewer.

7. **Secrets live in GitHub Secrets, never in YAML** — `${{ secrets.GCP_SA_KEY }}` not the key itself. Use Workload Identity Federation for production to eliminate static keys.

8. **Three environments, three configs** — dev (laptop/Cloud Shell), staging (full cloud, anonymized data), prod (real data, real dashboards). Never hardcode bucket names or dataset names.

9. **Prod deploys require manual approval** — staging auto-deploys on merge. Prod requires a human approver in GitHub Environment protection rules.

10. **Quality gates have teeth** — a gate that gets bypassed under pressure teaches the team quality is optional. Investigate root cause; never silently raise thresholds.

11. **GitOps: Git is the single source of truth** — no SSH uploads, no manual file copies. What's in Git is what runs in production. Every deploy has a PR, a reviewer, and an audit trail.

12. **Rollback = revert commit + new PR** — never `git push --force`. Create a revert commit, open a PR, let CI validate, merge, deploy. Rollback in 5 minutes with full audit trail.

---

### Homework: Deploy a Change via PR

**Complete this end-to-end before the next session:**

1. **Fork / clone** the `systems-in-production` repo
2. Create a branch: `git checkout -b feature/m10-cicd-homework`
3. **Add a unit test** in `tests/test_silver_transform.py` that checks `duration_min` is never negative
4. **Create `.github/workflows/ci.yml`** — at minimum: checkout, setup-python, pip install pytest, run pytest
5. **Open a Pull Request** — observe the CI workflow run in the Actions tab
6. **Make the test fail intentionally** (add a row with `duration_sec = -60` to your test fixture) → confirm CI blocks the PR
7. **Fix the test** → confirm CI passes → merge

**Stretch goal:**
- Add a second job to your CI workflow that validates your Airflow DAG imports
- Add a CDKTF `main.py` that creates one GCS bucket — run `cdktf synth` and commit the generated `cdktf.out/` snapshot

---

### What's Next

| Module | Topic |
|---|---|
| M11 | Streaming Data Pipelines (Pub/Sub, Kafka, Kinesis) |
| M12 | Data Governance and Cataloging (Dataplex, Glue Catalog) |
| M13 | Monitoring and Observability for Data Pipelines |
| M14 | Capstone: Full End-to-End Pipeline (Build → Orchestrate → CI/CD → Monitor) |

In [ ]:
# ============================================================
# M10 Complete -- Module Summary
# CI/CD = Continuous Integration / Continuous Deployment
# IaC   = Infrastructure as Code
# ============================================================
# WHY end with a summary cell?
# After 5+ hours of labs, students need a single-screen recap of what
# they built and what they can now do on the job.

# Section map: number -> (title, format, estimated time)
# These match the notebook section headers exactly so students can navigate.
sections = {
    1: ("Why CI/CD for Data Pipelines",         "Concept + simulation",    "20 min"),
    2: ("IaC Landscape",                         "Concept + comparison",    "25 min"),
    3: ("Terraform HCL -- Read and Understand",  "Lecture + HCL examples",  "30 min"),
    4: ("CDKTF -- Python IaC for GCP",           "Lab (Python code)",       "60 min"),
    5: ("AWS CDK -- Same Pipeline on AWS",       "Lab (Python code)",       "40 min"),
    6: ("GitHub Actions -- CI Pipeline",         "Lab (YAML + Python)",     "50 min"),
    7: ("Environment Promotion",                 "Concept + simulation",    "30 min"),
    8: ("Data Quality Gates in CI",              "Lab (Python tests)",      "30 min"),
    9: ("DAG Deployment -- GitOps for Airflow",  "Lab (YAML + simulation)", "30 min"),
}

# ── Print the module header ──────────────────────────────────────────────────
print("=" * 65)
print("  M10: CI/CD for Data Engineering")
print("  Program: Quintrix Jr. Data Engineer Training")
print("  Author:  Sunil Mogadati")
print("=" * 65)
print()

# ── Print section table with time estimates ──────────────────────────────────
print(f"  {'Sec':<4} {'Section':<40} {'Format':<25} {'Time'}")
print(f"  {'-'*4} {'-'*40} {'-'*25} {'-'*8}")
total_min = 0
for sec, (title, fmt, time_str) in sections.items():
    # Parse the integer minutes out of the "XX min" string
    mins = int(time_str.split()[0])
    total_min += mins
    print(f"  {sec:<4} {title:<40} {fmt:<25} {time_str}")

# Convert total minutes to hours + minutes for readability
hrs  = total_min // 60
mins = total_min % 60
print()
# You should see: total around 5h 35m
print(f"  Total estimated time: {total_min} min ({hrs}h {mins}m)")
print()

# ── Tools covered -- mapped to the sections where they appeared ──────────────
print("  Tools covered:")
tools = [
    # HCL = HashiCorp Configuration Language (Terraform native format, Section 3)
    "Terraform HCL (read-only fluency)",
    # CDKTF = CDK for Terraform -- Python generates Terraform JSON (Section 4)
    "CDKTF (Python -> Terraform JSON -> GCP API)",
    # CDK = Cloud Development Kit -- Python generates CloudFormation (Section 5)
    "AWS CDK (Python -> CloudFormation -> AWS API)",
    # GitHub Actions = CI/CD platform built into GitHub (Section 6)
    "GitHub Actions (CI/CD workflows)",
    # pytest = Python testing framework that GitHub Actions runs on every PR
    "pytest (unit tests for Silver transforms)",
    # ruff = fast Python linter (runs in under 1 second on large codebases)
    "ruff (Python linting)",
    # Great Expectations = data quality framework; QualityGate class is a
    # simplified in-notebook version of the same concept
    "Great Expectations / custom QualityGate (data quality)",
    # gsutil rsync / aws s3 sync = the actual deploy commands in the CD step
    "gsutil rsync / aws s3 sync (GitOps DAG deployment)",
    # CDKTF stacks with env parameter = how promotion config is managed
    "CDKTF stacks (environment promotion)",
]
for t in tools:
    print(f"    - {t}")
print()

# ── Outcomes: what a student can do after completing this module ─────────────
print("  What you can do after M10:")
print("    - Read any Terraform config at a job interview")
print("    - Provision your full pipeline infrastructure with Python")
print("    - Build a CI pipeline that blocks bad code before merge")
print("    - Deploy DAGs via PR (no SSH, no manual uploads)")
print("    - Gate production deploys with automated data quality checks")